# HuntCrew: agent-led threat detection and response

A crew of agents (CrewAI + RAG over MITRE ATT&CK) detects attacks in **replayed real network traffic**
(CIC-IDS2017), names the technique, plans a policy-checked response, asks a human before risky actions and
writes an incident report. **Grading is done by code against the dataset's own labels**, never by the agents.

**This one notebook is the whole project.** Every source file is written out by the cells below; nothing is
cloned. The repository (https://github.com/Paritsingla7/HuntCrew) holds the same files plus docs and tests.

**How to run (about 10-15 minutes, free CPU runtime is enough)**
1. Add a Colab Secret named `MISTRAL_API_KEY` (key icon in the left sidebar) and switch *Notebook access* on.
2. Click **Runtime -> Run all**. The first pass installs packages and then restarts the session on purpose
   (Colab says "session crashed"); click **Run all** again.
3. Click **Grant access** when Colab asks about the secret. If a risky action is proposed, answer the
   `approve? [y/N]` prompt under section 3.

## 1. Install packages (once)

In [ ]:
# Colab must restart its Python process after packages such as Pillow change, so on the first run this cell
# installs and then restarts the session; just run all cells again afterwards.
import importlib.util, os, signal, subprocess, sys
REQUIREMENTS = ['crewai==1.15.28', 'chromadb==1.5.9', 'litellm==1.103.1', 'sentence-transformers', 'PyYAML']
if importlib.util.find_spec("crewai") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS], check=True)
    print("Packages installed. Restarting the session now - run all cells again.")
    os.kill(os.getpid(), signal.SIGKILL)
print("dependencies ready")

## 2. The project's files
Each cell below writes one file of the project to disk, exactly as it is in the repository.

In [ ]:
import os
for d in ["src/huntcrew", "scripts", "data/policy", "results"]:
    os.makedirs(f"/content/HuntCrew/{d}", exist_ok=True)
%cd /content/HuntCrew

### Configuration, policy and saved results
`config.yaml` holds every tunable (model, throttle, pre-registered splits, label mapping, gate bar). `policy.json` is the action catalog and protected assets. `results/final.json` holds the numbers of the one frozen evaluation run.

In [ ]:
%%writefile /content/HuntCrew/config.yaml
# HuntCrew — single source of truth for tunables. Swap models/paths here, not in code.

llm:
  model: mistral/ministral-14b-latest   # LiteLLM model string; key from env MISTRAL_API_KEY
  temperature: 0.0
  max_tokens: 4000
  max_retries: 5
  min_interval_s: 2.1                   # measured via response headers: 30 requests/minute for this model
  tokens_per_minute: 700000             # measured: 937,500 tokens/minute; keep headroom
  cache_dir: .cache/llm

rag:
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  top_k: 5
  persist_dir: .cache/chroma

data:
  cic_parquet: data/external/cic-ids2017/CICIDS_Flow.parquet
  attack_stix: data/external/attack-stix/enterprise-attack.json
  policy: data/policy/policy.json

digest:
  max_private: 40              # every in-network source is listed up to this cap
  public_top_k: 6              # per neutral view (flows, ports, destinations, bytes, regularity)

replay:
  window_minutes: 5
  history_windows: 12          # how far back the agent's tools may look; never forward

labels:
  in_scope: [BENIGN, FTP-Patator, SSH-Patator, Bot, PortScan]
  # Hand-made mapping, published in docs/label-mapping.md. Grading is parent-level:
  # a truth of T1110 accepts T1110 or any T1110.xxx sub-technique.
  to_technique:
    FTP-Patator: T1110
    SSH-Patator: T1110
    Bot: T1071
    PortScan: T1046
  match_level: parent

# Pre-registered on 2026-10-15, before any detector result existed (ADR-019). Do not edit after results.
# Tune/dev choices (digest statistics, B0 thresholds) may use `dev` and `baseline_benign` only.
splits:
  baseline_benign: {start: "2017-07-03 09:00", end: "2017-07-03 17:00"}   # Monday: benign only
  dev:
    - {name: ftp_patator, start: "2017-07-04 09:00", end: "2017-07-04 10:45"}
  test:
    - {name: ssh_patator, start: "2017-07-04 14:00", end: "2017-07-04 15:30"}
    - {name: bot,         start: "2017-07-07 10:00", end: "2017-07-07 12:00"}
    - {name: portscan,    start: "2017-07-07 14:00", end: "2017-07-07 15:30"}

# B0 baseline only (a classic rules engine). The agent detector does NOT use these.
baseline_rules:                # evaluated per replay window
  password_guessing: {min_flows: 50, ports: [21, 22]}
  port_scan: {min_distinct_ports: 100}
  beaconing: {min_flows: 10, min_active_minutes: 3, max_cv: 0.3, require_rare_destination: true}

# Pre-registered bar for "agent detector is usable" (ADR-019). If missed -> B2-hybrid.
decision_gate:
  min_recall_vs_b0: 0.70       # agent recall >= 70% of B0 recall on dev
  max_fp_ratio_vs_b0: 2.0      # agent false-positive incidents <= 2x B0's

# Fixed before any detector result existed. A (window, entity) is attack-positive when it has at least
# this many attack-labelled flows from that source in the window.
grading:
  min_attack_flows: 5


In [ ]:
%%writefile /content/HuntCrew/data/policy/policy.json
{
  "_note": "Organisational policy (hand-made by design: it is a rule set, not a retrievable fact). Protected assets were derived from the data: the two hosts receiving almost all DNS (port 53) flows in CIC-IDS2017.",
  "protected_assets": [
    {"ip": "192.168.10.3", "role": "DNS / domain controller", "evidence": "663,674 dst-port-53 flows"},
    {"ip": "192.168.10.1", "role": "gateway / secondary DNS", "evidence": "294,066 dst-port-53 flows"}
  ],
  "action_catalog": {
    "block_ip":             {"risk": "low",    "description": "Block an IP address at the perimeter firewall"},
    "block_domain":         {"risk": "low",    "description": "Block or sinkhole a domain at the DNS resolver"},
    "open_ticket":          {"risk": "low",    "description": "Open a ticket for follow-up by a human"},
    "rate_limit_ip":        {"risk": "low",    "description": "Rate-limit connections from an IP"},
    "enable_waf_rule":      {"risk": "medium", "description": "Enable a web application firewall rule set"},
    "force_password_reset": {"risk": "medium", "description": "Force a password reset and revoke sessions for a user"},
    "disable_account":      {"risk": "medium", "description": "Disable a user account"},
    "isolate_host":         {"risk": "high",   "description": "Quarantine a host from the network"}
  }
}


In [ ]:
%%writefile /content/HuntCrew/results/final.json
{
  "split": "test",
  "model": "mistral/ministral-14b-latest",
  "frozen_commit": "19b8682",
  "slices": {
    "ssh_patator": {
      "windows": 18,
      "hybrid": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 20,
          "tp": 14,
          "fp": 6,
          "fn": 0,
          "precision": 0.7,
          "recall": 1.0,
          "f1": 0.824,
          "window_recall": 1.0,
          "technique_accuracy_on_tp": 0.714
        },
        "incidents": 4,
        "response": {
          "incidents": 4,
          "attack_linked_incidents": 1,
          "incident_precision": 0.25,
          "technique_accuracy_incident": 1.0,
          "grounding_rate": 1.0,
          "policy_pass_rate": 1.0,
          "mitigation_grounded_share": 0.5,
          "proposed_actions": 8
        },
        "environment": {
          "blocked_ips": 4,
          "rate_limited": 2,
          "blocked_domains": 0,
          "isolated_hosts": 0,
          "disabled_accounts": 0,
          "password_resets": 0,
          "waf_rules": 0,
          "tickets": 1
        },
        "llm": {
          "calls": 78,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 120938,
          "completion_tokens": 13424,
          "waited_s": 48.8
        },
        "seconds": 232.9
      },
      "b0": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 34,
          "tp": 14,
          "fp": 20,
          "fn": 0,
          "precision": 0.412,
          "recall": 1.0,
          "f1": 0.583,
          "window_recall": 1.0,
          "technique_accuracy_on_tp": 1.0
        },
        "incidents": 4,
        "llm": {
          "calls": 0,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 0,
          "completion_tokens": 0,
          "waited_s": 0.0
        },
        "seconds": 1.0
      },
      "b1": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 120,
          "tp": 0,
          "fp": 120,
          "fn": 14,
          "precision": 0.0,
          "recall": 0.0,
          "f1": 0.0,
          "window_recall": 0.0,
          "technique_accuracy_on_tp": 0.0
        },
        "incidents": 108,
        "llm": {
          "calls": 18,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 84674,
          "completion_tokens": 14816,
          "waited_s": 0.0
        },
        "seconds": 164.4
      },
      "agent": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 92,
          "tp": 11,
          "fp": 81,
          "fn": 3,
          "precision": 0.12,
          "recall": 0.786,
          "f1": 0.208,
          "window_recall": 0.786,
          "technique_accuracy_on_tp": 0.818
        },
        "incidents": 60,
        "llm": {
          "calls": 158,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 1013808,
          "completion_tokens": 74448,
          "waited_s": 40.4
        },
        "seconds": 977.8
      }
    },
    "bot": {
      "windows": 24,
      "hybrid": {
        "detection": {
          "truth_positives": 73,
          "predicted_positives": 25,
          "tp": 9,
          "fp": 16,
          "fn": 64,
          "precision": 0.36,
          "recall": 0.123,
          "f1": 0.184,
          "window_recall": 0.167,
          "technique_accuracy_on_tp": 0.0
        },
        "incidents": 14,
        "response": {
          "incidents": 14,
          "attack_linked_incidents": 7,
          "incident_precision": 0.5,
          "technique_accuracy_incident": 0.0,
          "grounding_rate": 1.0,
          "policy_pass_rate": 1.0,
          "mitigation_grounded_share": 0.516,
          "proposed_actions": 31
        },
        "environment": {
          "blocked_ips": 10,
          "rate_limited": 6,
          "blocked_domains": 0,
          "isolated_hosts": 0,
          "disabled_accounts": 0,
          "password_resets": 0,
          "waf_rules": 0,
          "tickets": 9
        },
        "llm": {
          "calls": 137,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 201460,
          "completion_tokens": 32216,
          "waited_s": 59.2
        },
        "seconds": 495.7
      },
      "b0": {
        "detection": {
          "truth_positives": 73,
          "predicted_positives": 38,
          "tp": 9,
          "fp": 29,
          "fn": 64,
          "precision": 0.237,
          "recall": 0.123,
          "f1": 0.162,
          "window_recall": 0.167,
          "technique_accuracy_on_tp": 1.0
        },
        "incidents": 13,
        "llm": {
          "calls": 0,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 0,
          "completion_tokens": 0,
          "waited_s": 0.0
        },
        "seconds": 1.2
      },
      "b1": {
        "detection": {
          "truth_positives": 73,
          "predicted_positives": 150,
          "tp": 2,
          "fp": 148,
          "fn": 71,
          "precision": 0.013,
          "recall": 0.027,
          "f1": 0.018,
          "window_recall": 0.083,
          "technique_accuracy_on_tp": 0.0
        },
        "incidents": 129,
        "llm": {
          "calls": 24,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 104101,
          "completion_tokens": 17889,
          "waited_s": 0.0
        },
        "seconds": 198.7
      },
      "agent": {
        "detection": {
          "truth_positives": 73,
          "predicted_positives": 123,
          "tp": 18,
          "fp": 105,
          "fn": 55,
          "precision": 0.146,
          "recall": 0.247,
          "f1": 0.184,
          "window_recall": 0.458,
          "technique_accuracy_on_tp": 0.111
        },
        "incidents": 81,
        "llm": {
          "calls": 202,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 1146456,
          "completion_tokens": 87774,
          "waited_s": 46.8
        },
        "seconds": 1119.3
      }
    },
    "portscan": {
      "windows": 18,
      "hybrid": {
        "detection": {
          "truth_positives": 8,
          "predicted_positives": 23,
          "tp": 5,
          "fp": 18,
          "fn": 3,
          "precision": 0.217,
          "recall": 0.625,
          "f1": 0.323,
          "window_recall": 0.625,
          "technique_accuracy_on_tp": 0.0
        },
        "incidents": 9,
        "response": {
          "incidents": 9,
          "attack_linked_incidents": 1,
          "incident_precision": 0.111,
          "technique_accuracy_incident": 0.0,
          "grounding_rate": 1.0,
          "policy_pass_rate": 1.0,
          "mitigation_grounded_share": 0.647,
          "proposed_actions": 17
        },
        "environment": {
          "blocked_ips": 7,
          "rate_limited": 5,
          "blocked_domains": 0,
          "isolated_hosts": 0,
          "disabled_accounts": 0,
          "password_resets": 0,
          "waf_rules": 0,
          "tickets": 1
        },
        "llm": {
          "calls": 110,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 159060,
          "completion_tokens": 22907,
          "waited_s": 65.3
        },
        "seconds": 359.1
      },
      "b0": {
        "detection": {
          "truth_positives": 8,
          "predicted_positives": 32,
          "tp": 5,
          "fp": 27,
          "fn": 3,
          "precision": 0.156,
          "recall": 0.625,
          "f1": 0.25,
          "window_recall": 0.625,
          "technique_accuracy_on_tp": 1.0
        },
        "incidents": 9,
        "llm": {
          "calls": 0,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 0,
          "completion_tokens": 0,
          "waited_s": 0.0
        },
        "seconds": 1.1
      },
      "b1": {
        "detection": {
          "truth_positives": 8,
          "predicted_positives": 134,
          "tp": 1,
          "fp": 133,
          "fn": 7,
          "precision": 0.007,
          "recall": 0.125,
          "f1": 0.014,
          "window_recall": 0.125,
          "technique_accuracy_on_tp": 1.0
        },
        "incidents": 118,
        "llm": {
          "calls": 18,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 80668,
          "completion_tokens": 16877,
          "waited_s": 0.0
        },
        "seconds": 171.2
      },
      "agent": {
        "detection": {
          "truth_positives": 8,
          "predicted_positives": 92,
          "tp": 5,
          "fp": 87,
          "fn": 3,
          "precision": 0.054,
          "recall": 0.625,
          "f1": 0.1,
          "window_recall": 0.625,
          "technique_accuracy_on_tp": 0.0
        },
        "incidents": 67,
        "llm": {
          "calls": 158,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 979506,
          "completion_tokens": 73679,
          "waited_s": 44.4
        },
        "seconds": 954.6
      }
    }
  },
  "micro_average": {
    "b0": {
      "tp": 28,
      "fp": 76,
      "fn": 67,
      "precision": 0.269,
      "recall": 0.295,
      "f1": 0.281
    },
    "b1": {
      "tp": 3,
      "fp": 401,
      "fn": 92,
      "precision": 0.007,
      "recall": 0.032,
      "f1": 0.012
    },
    "agent": {
      "tp": 34,
      "fp": 273,
      "fn": 61,
      "precision": 0.111,
      "recall": 0.358,
      "f1": 0.169
    },
    "hybrid": {
      "tp": 28,
      "fp": 40,
      "fn": 67,
      "precision": 0.412,
      "recall": 0.295,
      "f1": 0.344
    }
  },
  "hybrid_response": {
    "incidents": 27,
    "attack_linked_incidents": 9,
    "technique_correct": 1,
    "technique_accuracy_incident": 0.111,
    "grounding_rate": 1.0,
    "policy_pass_rate": 1.0
  },
  "dev": {
    "ftp_patator": {
      "windows": 21,
      "hybrid": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 31,
          "tp": 13,
          "fp": 18,
          "fn": 1,
          "precision": 0.419,
          "recall": 0.929,
          "f1": 0.578,
          "window_recall": 0.929,
          "technique_accuracy_on_tp": 0.077
        },
        "incidents": 8,
        "response": {
          "incidents": 8,
          "attack_linked_incidents": 1,
          "incident_precision": 0.125,
          "technique_accuracy_incident": 0.0,
          "grounding_rate": 0.917,
          "policy_pass_rate": 1.0,
          "mitigation_grounded_share": 0.579,
          "proposed_actions": 19
        },
        "environment": {
          "blocked_ips": 7,
          "rate_limited": 5,
          "blocked_domains": 0,
          "isolated_hosts": 0,
          "disabled_accounts": 0,
          "password_resets": 0,
          "waf_rules": 0,
          "tickets": 3
        },
        "llm": {
          "calls": 32,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 28983,
          "completion_tokens": 9192,
          "waited_s": 6.9
        },
        "seconds": 124.3
      },
      "b0": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 48,
          "tp": 13,
          "fp": 35,
          "fn": 1,
          "precision": 0.271,
          "recall": 0.929,
          "f1": 0.419,
          "window_recall": 0.929,
          "technique_accuracy_on_tp": 1.0
        },
        "incidents": 9,
        "llm": {
          "calls": 0,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 0,
          "completion_tokens": 0,
          "waited_s": 0.0
        },
        "seconds": 1.1
      },
      "b1": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 124,
          "tp": 3,
          "fp": 121,
          "fn": 11,
          "precision": 0.024,
          "recall": 0.214,
          "f1": 0.043,
          "window_recall": 0.214,
          "technique_accuracy_on_tp": 0.333
        },
        "incidents": 113,
        "llm": {
          "calls": 21,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 93826,
          "completion_tokens": 16079,
          "waited_s": 0.0
        },
        "seconds": 177.3
      },
      "agent": {
        "detection": {
          "truth_positives": 14,
          "predicted_positives": 104,
          "tp": 9,
          "fp": 95,
          "fn": 5,
          "precision": 0.087,
          "recall": 0.643,
          "f1": 0.153,
          "window_recall": 0.643,
          "technique_accuracy_on_tp": 0.333
        },
        "incidents": 63,
        "llm": {
          "calls": 0,
          "cache_hits": 0,
          "retries": 0,
          "prompt_tokens": 0,
          "completion_tokens": 0,
          "waited_s": 0.0
        },
        "seconds": 0.0
      }
    }
  }
}


### The `huntcrew` package
`schema`, `config`: data contracts and settings. `data_cic`, `replay`, `digest`, `profile`: real traffic, one window at a time, summarised as statistics. `attack_kb`: ATT&CK knowledge base (RAG). `detector_tools`, `evidence`, `detector`: the detector agent and its code verification. `crew`, `pipeline`, `response`: technique naming, response planning, policy, simulated environment, approval gate. `baselines`, `evaluate`: the comparison systems and the grader (code only). `llm`, `syncutil`, `demo`: model plumbing and notebook helpers.

In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/__init__.py



In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/attack_kb.py
"""ATT&CK knowledge base: official STIX -> one chunk per (sub-)technique -> embeddings -> ChromaDB.

Design points (docs/DECISIONS.md ADR-003, ADR-021):
  * built from the official MITRE STIX bundle, nothing hand-written;
  * each chunk joins the technique with its official mitigations and detection analytics;
  * every search is recorded in `trace`, so guardrails can verify a cited ID was really retrieved.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

_CITATION = re.compile(r"\(Citation:[^)]*\)")
_TAGS = re.compile(r"</?code>|<br\s*/?>")
_MD_LINK = re.compile(r"\[([^\]]+)\]\([^)]+\)")


def clean(text: str) -> str:
    text = _CITATION.sub("", text or "")
    text = _TAGS.sub(" ", text)
    text = _MD_LINK.sub(r"\1", text)
    return re.sub(r"\s+", " ", text).strip()


@dataclass(frozen=True)
class Mitigation:
    id: str
    name: str
    how: str


@dataclass
class Chunk:
    id: str                       # ATT&CK ID, e.g. T1110 or T1110.001
    name: str
    tactics: list[str]
    is_subtechnique: bool
    parent_id: str | None
    platforms: list[str]
    description: str
    detection: list[str]          # analytic descriptions
    mitigations: list[Mitigation]

    def text(self, max_description: int = 900) -> str:
        """Text that gets embedded. Most informative content first (embedding models truncate)."""
        parts = [f"{self.name} ({self.id}). Tactics: {', '.join(self.tactics)}.",
                 self.description[:max_description]]
        if self.detection:
            parts.append("Detection: " + " ".join(self.detection[:3]))
        if self.mitigations:
            parts.append("Mitigations: " + ", ".join(m.name for m in self.mitigations))
        return " ".join(parts)


@dataclass
class Hit:
    id: str
    name: str
    score: float                  # cosine similarity, higher is better
    tactics: list[str]
    snippet: str
    mitigations: list[Mitigation] = field(default_factory=list)


def _attack_id(obj: dict) -> str | None:
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None


def _live(obj: dict) -> bool:
    return not obj.get("revoked") and not obj.get("x_mitre_deprecated")


def load_chunks(stix_path: str | Path) -> list[Chunk]:
    with open(stix_path, encoding="utf-8") as f:
        objs = json.load(f)["objects"]
    by_id = {o["id"]: o for o in objs}

    mitig: dict[str, list[Mitigation]] = {}
    detect: dict[str, list[str]] = {}
    parent_of: dict[str, str] = {}
    for r in objs:
        if r["type"] != "relationship" or r.get("revoked"):
            continue
        src, tgt = by_id.get(r["source_ref"]), by_id.get(r["target_ref"])
        if not src or not tgt:
            continue
        kind = r["relationship_type"]
        if kind == "mitigates" and src["type"] == "course-of-action" and _live(src):
            mitig.setdefault(tgt["id"], []).append(
                Mitigation(_attack_id(src) or "", src["name"], clean(r.get("description", ""))[:300]))
        elif kind == "detects" and src["type"] == "x-mitre-detection-strategy" and _live(src):
            for ref in src.get("x_mitre_analytic_refs", []):
                a = by_id.get(ref)
                if a and a.get("description"):
                    detect.setdefault(tgt["id"], []).append(clean(a["description"]))
        elif kind == "subtechnique-of":
            parent_of[src["id"]] = tgt["id"]

    chunks: list[Chunk] = []
    for o in objs:
        if o["type"] != "attack-pattern" or not _live(o):
            continue
        aid = _attack_id(o)
        if not aid:
            continue
        tactics = [p["phase_name"] for p in o.get("kill_chain_phases", [])
                   if p.get("kill_chain_name") == "mitre-attack"]
        parent = by_id.get(parent_of.get(o["id"], ""))
        own = {m.id: m for m in mitig.get(o["id"], [])}
        chunks.append(Chunk(
            id=aid, name=o["name"], tactics=tactics,
            is_subtechnique=bool(o.get("x_mitre_is_subtechnique")),
            parent_id=_attack_id(parent) if parent else None,
            platforms=o.get("x_mitre_platforms", []),
            description=clean(o.get("description", "")),
            detection=list(dict.fromkeys(detect.get(o["id"], []))),
            mitigations=sorted(own.values(), key=lambda m: m.id),
        ))
    chunks.sort(key=lambda c: c.id)
    return chunks


class AttackKB:
    def __init__(self, chunks: list[Chunk], embedding_model: str, persist_dir: str | Path | None = None):
        import chromadb
        from sentence_transformers import SentenceTransformer

        self.chunks = {c.id: c for c in chunks}
        self.model = SentenceTransformer(embedding_model)
        self.trace: list[dict] = []                  # one entry per search: query + returned IDs

        client = chromadb.PersistentClient(path=str(persist_dir)) if persist_dir else chromadb.EphemeralClient()
        name = "attack_" + re.sub(r"\W+", "_", embedding_model)[-40:]
        existing = {c.name for c in client.list_collections()}
        if name in existing:
            col = client.get_collection(name)
            if col.count() != len(chunks):
                client.delete_collection(name)
                existing.discard(name)
        if name not in existing:
            col = client.create_collection(name, metadata={"hnsw:space": "cosine"})
            texts = [c.text() for c in chunks]
            emb = self.model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
            col.add(ids=[c.id for c in chunks], embeddings=emb.tolist(),
                    documents=texts, metadatas=[{"name": c.name, "sub": c.is_subtechnique} for c in chunks])
        self.col = client.get_collection(name)

    def search(self, query: str, k: int = 5, include_subtechniques: bool = True) -> list[Hit]:
        q = self.model.encode([query], normalize_embeddings=True, show_progress_bar=False)
        where: dict[str, Any] | None = None if include_subtechniques else {"sub": False}
        res = self.col.query(query_embeddings=q.tolist(), n_results=k,
                             where=where)
        dists = (res["distances"] or [[]])[0]
        hits = []
        for cid, dist in zip(res["ids"][0], dists, strict=True):
            c = self.chunks[cid]
            hits.append(Hit(id=c.id, name=c.name, score=round(1.0 - float(dist), 4), tactics=c.tactics,
                            snippet=c.description[:400], mitigations=c.mitigations))
        self.trace.append({"query": query, "ids": [h.id for h in hits], "scores": [h.score for h in hits]})
        return hits

    def retrieved_ids(self) -> set[str]:
        return {i for t in self.trace for i in t["ids"]}

    def get(self, attack_id: str) -> Chunk | None:
        return self.chunks.get(attack_id)

    def mitigations_for(self, attack_id: str) -> list[Mitigation]:
        """Official mitigations for a technique; falls back to the parent technique for sub-techniques."""
        c = self.chunks.get(attack_id)
        if not c:
            return []
        if c.mitigations or not c.parent_id:
            return c.mitigations
        parent = self.chunks.get(c.parent_id)
        return parent.mitigations if parent else []


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/baselines.py
"""B0: a classic rules engine with a fixed rule -> technique map (what a basic SIEM does).

These thresholds are hand-made on purpose (see docs/03-architecture.md, "What is still hand-made").
The agent detector does not use them; B0 exists to be beaten, or to show the agent cannot beat it.
"""
from __future__ import annotations

from typing import cast

import pandas as pd

from .digest import is_private
from .replay import Window
from .schema import Finding, TechniqueHint

RULE_TO_TECHNIQUE = {"password_guessing": "T1110", "port_scan": "T1046", "beaconing": "T1071"}


def _finding(window: Window, entity: str, rule: str, behaviour: str, evidence: dict[str, float]) -> Finding:
    return Finding(window_start=window.start, window_end=window.end, entity=entity, behaviour=behaviour,
                   evidence=evidence, confidence=1.0, suspicious=True,
                   technique_hints=[TechniqueHint(id=RULE_TO_TECHNIQUE[rule], citation=f"b0-rule:{rule}")])


def known_destinations(baseline_flows: pd.DataFrame) -> set[str]:
    """External destinations seen in benign baseline traffic (used by the rarity condition)."""
    d = baseline_flows["dst_ip"].drop_duplicates()
    return {str(i) for i in d if not is_private(str(i))}


def b0_findings(current: pd.DataFrame, window: Window, rules: dict,
                known_dsts: set[str] | None = None) -> list[Finding]:
    if current.empty:
        return []
    out: list[Finding] = []

    pg = rules["password_guessing"]
    svc = current[current["dst_port"].isin(pg["ports"])]
    counts_df = svc.groupby(["src_ip", "dst_port"]).size().reset_index(name="n")
    for row in counts_df[counts_df["n"] >= pg["min_flows"]].itertuples(index=False):
        src, port, n = str(row.src_ip), int(row.dst_port), int(row.n)
        out.append(_finding(window, src, "password_guessing",
                            f"{n} flows to port {port}", {"flows": float(n), "dst_port": float(port)}))

    ps = rules["port_scan"]
    # distinct ports on ONE destination host; a server replying to many clients is not scanning
    pair = current.groupby(["src_ip", "dst_ip"])["dst_port"].nunique().reset_index(name="n")
    for row in pair[pair["n"] >= ps["min_distinct_ports"]].itertuples(index=False):
        out.append(_finding(window, str(row.src_ip), "port_scan",
                            f"{int(row.n)} distinct destination ports on {row.dst_ip}",
                            {"distinct_ports": float(row.n)}))

    bc = rules["beaconing"]
    ext = current[~current["dst_ip"].map(is_private)]
    if bc.get("require_rare_destination") and known_dsts is not None:
        ext = ext[~ext["dst_ip"].isin(known_dsts)]      # only destinations never seen in normal traffic
    if not ext.empty:
        minute = ext["ts"].dt.floor("min")
        per_min = (ext.assign(_m=minute).groupby(["src_ip", "dst_ip", "dst_port", "_m"])
                   .size().reset_index(name="n"))
        for key, grp in per_min.groupby(["src_ip", "dst_ip", "dst_port"]):
            bsrc, bdst, bport = (str(k) for k in cast(tuple, key))
            counts = grp["n"]
            if len(counts) >= bc["min_active_minutes"] and counts.sum() >= bc["min_flows"]:
                mean = counts.mean()
                cv = counts.std(ddof=0) / mean if mean else 0.0
                if cv <= bc["max_cv"]:
                    out.append(_finding(window, bsrc, "beaconing", f"regular flows to {bdst}:{bport}",
                                        {"flows": float(counts.sum()), "cv": float(round(cv, 3)),
                                         "dst_port": float(bport)}))
    return out


B1_PROMPT = """You are a network security analyst. Below are statistics for ONE {minutes}-minute window of
network flows ({label}). List the sources that look like attacks and, for each, the MITRE ATT&CK technique
ID you believe applies (from your own knowledge). If nothing is suspicious return {{"findings": []}}.

Answer with ONLY this JSON:
{{"findings": [{{"entity": "<ip>", "behaviour": "<=20 words", "technique_hints": [{{"id": "T0000"}}],
"confidence": 0.0}}]}}

{digest}
"""


def b1_window(llm, digest_text: str, window: Window, minutes: int) -> list[Finding]:
    """B1: one prompt, no tools, no retrieval, no baseline. Entities are not evidence-checked (it has none)."""
    import json

    text = str(llm.call([{"role": "user", "content": B1_PROMPT.format(
        minutes=minutes, label=window.label(), digest=digest_text)}]))
    i, j = text.find("{"), text.rfind("}")
    try:
        data = json.loads(text[i:j + 1]) if i >= 0 and j > i else {}
    except json.JSONDecodeError:
        return []
    out: list[Finding] = []
    for d in (data.get("findings") or [])[:8]:
        if not isinstance(d, dict) or not d.get("entity"):
            continue
        hints = [TechniqueHint(id=str(h.get("id", "")).strip(), citation="b1-memory")
                 for h in (d.get("technique_hints") or []) if isinstance(h, dict) and h.get("id")]
        out.append(Finding(window_start=window.start, window_end=window.end, entity=str(d["entity"]),
                           behaviour=str(d.get("behaviour", ""))[:200], technique_hints=hints,
                           confidence=float(d.get("confidence", 0.5) or 0.5), suspicious=True))
    return out


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/config.py
"""Config loader. Lookup order: explicit path -> $HUNTCREW_CONFIG -> ./config.yaml -> repo root."""
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import yaml

_REPO_ROOT = Path(__file__).resolve().parents[2]


def find_config(path: str | None = None) -> Path:
    candidates = [path, os.environ.get("HUNTCREW_CONFIG"), "config.yaml", _REPO_ROOT / "config.yaml"]
    for c in candidates:
        if c and Path(c).is_file():
            return Path(c)
    raise FileNotFoundError("config.yaml not found (set HUNTCREW_CONFIG or run from the project root)")


def load_config(path: str | None = None) -> dict[str, Any]:
    with open(find_config(path), encoding="utf-8") as f:
        return yaml.safe_load(f)


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/crew.py
"""Downstream agents: Threat-Intel, Strategist, Reporter. Code guardrails check every output.

Guardrails (plain code, ADR-007):
  * a technique must exist in ATT&CK AND have been retrieved in this incident's searches;
  * an action must be in the catalog; its mitigation must be an official mitigation of an assessed technique;
  * anything that fails is dropped and counted, so hallucination rates are measurable.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from typing import Any

os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

from .attack_kb import AttackKB  # noqa: E402
from .response import Policy  # noqa: E402
from .schema import Incident, ProposedAction, ResponsePlan, TechniqueAssessment, ThreatAssessment  # noqa: E402
from .syncutil import run_blocking  # noqa: E402

SEVERITIES = {"low", "medium", "high", "critical"}


def extract_json(text: str) -> dict | None:
    i, j = text.find("{"), text.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        data = json.loads(text[i:j + 1])
    except json.JSONDecodeError:
        return None
    return data if isinstance(data, dict) else None


def extract_list(text: str, key: str) -> list | None:
    """List under `key` in a JSON object, or a bare JSON array (models often drop the wrapper)."""
    obj = extract_json(text)
    if obj is not None and isinstance(obj.get(key), list):
        return obj[key]
    i, j = text.find("["), text.rfind("]")
    if i >= 0 and j > i:
        try:
            data = json.loads(text[i:j + 1])
        except json.JSONDecodeError:
            return None
        return data if isinstance(data, list) else None
    return None


def incident_brief(inc: Incident) -> str:
    lines = [f"Incident {inc.id}: source {inc.entity}, {inc.start:%Y-%m-%d %H:%M} to {inc.end:%H:%M}, "
             f"{len(inc.findings)} window(s) flagged."]
    for f in inc.findings[:6]:
        ev = ", ".join(f"{k}={v:g}" for k, v in f.evidence.items())
        lines.append(f"- {f.window_start:%H:%M}: {f.behaviour} [{ev}]")
    return "\n".join(lines)


def incident_profile(flows: Any, inc: Incident) -> str:
    """Neutral aggregate facts for an incident, computed by code from the flows the incident spans.

    No labels, no verdicts: just the quantities an analyst would read to tell different behaviours apart.
    """
    cur = flows[(flows["ts"] >= inc.start) & (flows["ts"] < inc.end) & (flows["src_ip"] == inc.entity)]
    n = len(cur)
    if n == 0:
        return "no flows"
    minutes = max(1.0, (inc.end - inc.start).total_seconds() / 60.0)
    unanswered = float((cur["bwd_pkts"] == 0).mean())
    ports = cur["dst_port"].value_counts()
    top = ", ".join(f"{int(p)}:{c / n:.0%}" for p, c in ports.head(3).items())
    return (f"Aggregate facts for {inc.entity} over the incident: flows={n}, flows_per_minute={n / minutes:.1f}, "
            f"distinct_dst_ips={cur['dst_ip'].nunique()}, distinct_dst_ports={cur['dst_port'].nunique()}, "
            f"top_dst_ports(share)={top}, unanswered_flow_share={unanswered:.2f} (flows with no backward packets), "
            f"mean_fwd_packets_per_flow={cur['fwd_pkts'].mean():.1f}, mean_bwd_packets_per_flow={cur['bwd_pkts'].mean():.1f}, "
            f"mean_fwd_bytes_per_flow={cur['fwd_bytes'].mean():.0f}, mean_bwd_bytes_per_flow={cur['bwd_bytes'].mean():.0f}, "
            f"mean_flow_duration_ms={cur['duration_us'].mean() / 1000:.1f}")


# --- guardrails -------------------------------------------------------------------------------
def guard_assessment(draft: dict, incident_id: str, kb_ids: set[str], retrieved: set[str]) -> ThreatAssessment:
    techs: list[TechniqueAssessment] = []
    dropped: list[str] = []
    for t in draft.get("techniques") or []:
        tid = str((t or {}).get("id", "")).strip()
        if not tid:
            continue
        if tid in kb_ids and tid in retrieved:
            try:
                conf = max(0.0, min(1.0, float(t.get("confidence", 0.5))))
            except (TypeError, ValueError):
                conf = 0.5
            techs.append(TechniqueAssessment(id=tid, confidence=conf, citation=str(t.get("citation", tid))))
        else:
            dropped.append(tid)
    return ThreatAssessment(incident_id=incident_id, techniques=techs,
                            rationale=str(draft.get("rationale", ""))[:600], dropped_ids=dropped)


@dataclass
class PlanGuard:
    plan: ResponsePlan
    dropped_actions: list[str] = field(default_factory=list)       # unknown action types
    ungrounded_mitigations: int = 0                                # mitigation id not official for the technique


def guard_plan(draft: dict, incident_id: str, assessment: ThreatAssessment, kb: AttackKB,
               policy: Policy) -> PlanGuard:
    official: set[str] = set()
    for t in assessment.techniques:
        official |= {m.id for m in kb.mitigations_for(t.id)}
    actions: list[ProposedAction] = []
    dropped: list[str] = []
    ungrounded = 0
    for a in draft.get("actions") or []:
        typ = str((a or {}).get("type", "")).strip()
        if typ not in policy.catalog:
            dropped.append(typ or "<empty>")
            continue
        mit = a.get("mitigation_id")
        mit = str(mit).strip() if mit else None
        if mit and mit not in official:
            ungrounded += 1
            mit = None
        actions.append(ProposedAction(type=typ, target=str(a.get("target", "")).strip(),
                                      justification=str(a.get("justification", ""))[:200], mitigation_id=mit))
    sev = str(draft.get("severity", "medium")).lower()
    try:
        risk = max(0, min(100, int(draft.get("risk_score", 50))))
    except (TypeError, ValueError):
        risk = 50
    plan = ResponsePlan(incident_id=incident_id, severity=sev if sev in SEVERITIES else "medium",
                        risk_score=risk, rationale=str(draft.get("rationale", ""))[:400], actions=actions)
    return PlanGuard(plan, dropped, ungrounded)


# --- agents -----------------------------------------------------------------------------------
INTEL_PROMPT = """Identify which MITRE ATT&CK technique(s) this incident corresponds to.

{brief}
{profile}

Steps:
1. Read the aggregate facts: are flows answered or unanswered, how large is each flow, how concentrated is
   the traffic on one port or destination, how regular is it, how many destinations or ports are touched.
2. Form at least THREE competing hypotheses that span different ATT&CK tactics (for example access or
   credential-related, discovery, command and control, exfiltration, impact). Call search_attack_kb once
   per hypothesis with a plain description of what the facts would look like under it.
3. Compare each hypothesis with the facts and choose the best-matching technique ID(s). Prefer the most
   specific sub-technique only if the evidence supports it; otherwise use the parent technique. At most 2.
Only cite IDs returned by search_attack_kb.

Answer with ONLY this JSON:
{{"techniques": [{{"id": "T0000", "confidence": 0.0, "citation": "<technique name>"}}],
"rationale": "<=40 words tying evidence to the technique"}}
"""

STRATEGY_PROMPT = """Plan the response to this incident.

{brief}
Assessed techniques: {techniques}

Allowed action types (type: risk): {catalog}
Official ATT&CK mitigations you may cite (id: name): {mitigations}

Rules: at most 4 actions; choose only action types listed above; use "mitigation_id" only from the
official list and only when the action implements it (otherwise null); the target of block_ip/rate_limit_ip/
isolate_host must be an IP address that appears in the brief. Never target these protected addresses: {protected}.

Answer with ONLY this JSON:
{{"severity": "low|medium|high|critical", "risk_score": 0, "rationale": "<=40 words",
"actions": [{{"type": "block_ip", "target": "<ip>", "justification": "<=20 words", "mitigation_id": null}}]}}
"""

JUDGE_PROMPT = """You are deciding which MITRE ATT&CK technique best explains an incident. Judge ONLY from the
facts below and the technique descriptions given; do not rely on memory.

{brief}
{profile}

CANDIDATE TECHNIQUES (from retrieval)
{candidates}

For each candidate, state which facts it explains and which facts it conflicts with, then score 0-10 for how
well its description explains ALL the facts together.
- Network flows never show payloads, credentials or login results. The absence of such direct evidence is
  NOT a contradiction; judge a technique by whether its typical network footprint (volume, concentration,
  regularity, answered or unanswered traffic, size per flow) matches the facts.
- Penalise only real conflicts: a technique about moving large volumes of data when each flow carries only
  a few bytes, or one that requires unanswered traffic when most flows were answered.

Answer with ONLY this JSON, one entry per candidate:
{{"scores": [{{"id": "T0000", "supports": "<=15 words", "contradicts": "<=15 words", "score": 0}}]}}
"""

REPORT_PROMPT = """Write a concise incident report in Markdown for a security manager.

{brief}
Assessed techniques: {techniques}
Severity: {severity} (risk score {risk})
Response log: {log}

Sections: Summary, Evidence, ATT&CK mapping, Actions taken (executed / awaiting approval / denied /
rejected), Recommendations. Do not invent facts beyond what is given.
"""


def _run(llm: Any, role: str, goal: str, backstory: str, tools: list[Any], prompt: str, max_iter: int = 5) -> str:
    from crewai import Agent, Crew, Process, Task

    agent = Agent(role=role, goal=goal, backstory=backstory, tools=tools, llm=llm, max_iter=max_iter,
                  allow_delegation=False, verbose=False)
    task = Task(description=prompt, expected_output="Exactly the format requested.", agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False)
    out = run_blocking(crew.kickoff)
    return str(getattr(out, "raw", out))


class DownstreamCrew:
    def __init__(self, kb: AttackKB, policy: Policy, llm: Any):
        self.kb, self.policy, self.llm = kb, policy, llm
        self.kb_ids = set(kb.chunks)

    def _search_tool(self) -> Any:
        from crewai.tools import tool

        kb = self.kb

        @tool("search_attack_kb")
        def search_attack_kb_tool(query: str) -> str:
            """Search the MITRE ATT&CK knowledge base by meaning. Returns technique IDs you may cite."""
            hits = kb.search(query, k=5)
            return "\n".join(f"{h.id} {h.name} [{', '.join(h.tactics)}] score={h.score:.2f}: {h.snippet[:160]}"
                             for h in hits)

        return search_attack_kb_tool

    def _candidates(self, start: int, agent_choice: list[str], k: int = 8, per_query: int = 2) -> list[str]:
        """Candidates for the judge: the best `per_query` results of EVERY hypothesis search (so one
        hypothesis cannot crowd out the others), then the agent's own picks, then the rest by score."""
        best: dict[str, float] = {}
        out: list[str] = []
        for t in self.kb.trace[start:]:
            scores = t.get("scores", [0.0] * len(t["ids"]))
            for tid, sc in zip(t["ids"], scores, strict=False):
                best[tid] = max(best.get(tid, 0.0), sc)
            for tid in t["ids"][:per_query]:
                if tid not in out:
                    out.append(tid)
        out += [t for t in agent_choice if t in best and t not in out]
        out += [t for t in sorted(best, key=lambda x: best[x], reverse=True) if t not in out]
        return out[:k]

    def assess(self, inc: Incident, profile: str = "") -> tuple[ThreatAssessment, str]:
        start = len(self.kb.trace)
        raw = _run(self.llm, "Threat intelligence analyst",
                   "Map the incident to the correct MITRE ATT&CK technique using retrieved evidence.",
                   "Knows ATT&CK well and never cites an ID without retrieving it.",
                   [self._search_tool()], INTEL_PROMPT.format(brief=incident_brief(inc), profile=profile))
        retrieved = {i for t in self.kb.trace[start:] for i in t["ids"]}
        draft = extract_json(raw) or {}
        first = guard_assessment(draft, inc.id, self.kb_ids, retrieved)
        cands = self._candidates(start, [t.id for t in first.techniques])
        if len(cands) < 2:
            return first, raw
        # judge step: score every retrieved candidate against the facts using the real ATT&CK descriptions
        desc = "\n".join(f"- {c}: {self.kb.chunks[c].name} [{', '.join(self.kb.chunks[c].tactics)}]: "
                         f"{self.kb.chunks[c].description[:380]}" for c in cands)
        judged = _run(self.llm, "ATT&CK technique judge",
                      "Score candidate techniques against the incident facts, using only the descriptions given.",
                      "Methodical; checks each fact against each description before scoring.", [],
                      JUDGE_PROMPT.format(brief=incident_brief(inc), profile=profile, candidates=desc), max_iter=2)
        scored = extract_list(judged, "scores")
        if not isinstance(scored, list):
            return first, raw + "\n[judge unparseable; used agent choice]\n" + judged[:1500]
        rows = []
        for s in scored:
            try:
                tid, sc = str(s.get("id", "")).strip(), float(s.get("score", 0))
            except (TypeError, ValueError, AttributeError):
                continue
            if tid in cands:
                rows.append((sc, tid))
        rows.sort(key=lambda r: (-r[0], cands.index(r[1])))
        top = [(sc, tid) for sc, tid in rows if sc >= 5][:2]
        if not top:
            return first, raw + "\n[judge found no candidate >= 5; used agent choice]"
        techs = [{"id": tid, "confidence": round(sc / 10, 2), "citation": self.kb.chunks[tid].name}
                 for sc, tid in top]
        final = guard_assessment({"techniques": techs, "rationale": first.rationale}, inc.id, self.kb_ids,
                                 set(cands) | retrieved)
        return final, raw + "\n[judge] " + judged

    def plan(self, inc: Incident, assessment: ThreatAssessment, profile: str = "") -> tuple[PlanGuard, str]:
        mits: dict[str, str] = {}
        for t in assessment.techniques:
            for m in self.kb.mitigations_for(t.id):
                mits[m.id] = m.name
        prompt = STRATEGY_PROMPT.format(
            brief=incident_brief(inc) + ("\n" + profile if profile else ""),
            techniques=", ".join(f"{t.id}" for t in assessment.techniques) or "none identified",
            catalog="; ".join(f"{k}: {v['risk']}" for k, v in self.policy.catalog.items()),
            mitigations="; ".join(f"{k}: {v}" for k, v in mits.items()) or "none available",
            protected=", ".join(sorted(self.policy.protected)) or "none")
        raw = _run(self.llm, "Incident response strategist",
                   "Propose a safe, policy-compliant response grounded in official ATT&CK mitigations.",
                   "Prefers containment that is proportionate, never touches protected assets.", [], prompt)
        return guard_plan(extract_json(raw) or {}, inc.id, assessment, self.kb, self.policy), raw

    def report(self, inc: Incident, assessment: ThreatAssessment, plan: ResponsePlan, log_text: str) -> str:
        return _run(self.llm, "Incident report writer", "Write a clear, factual incident report.",
                    "Writes for busy managers; states only what the data supports.", [],
                    REPORT_PROMPT.format(brief=incident_brief(inc),
                                         techniques=", ".join(t.id for t in assessment.techniques) or "none",
                                         severity=plan.severity, risk=plan.risk_score, log=log_text))


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/data_cic.py
"""CIC-IDS2017 loader.

Two things matter here:
  1. Timestamps in this dataset are messy (see `parse_timestamps`).
  2. The answer key (attack label) is returned as a SEPARATE object from the flows, so the
     detector side of the system never has a label column to peek at.
"""
from __future__ import annotations

from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

# parquet column -> canonical name. Only what the digest / tools need.
COLUMN_MAP = {
    "flow_id": "flow_id",
    "source_ip": "src_ip",
    "source_port": "src_port",
    "destination_ip": "dst_ip",
    "destination_port": "dst_port",
    "protocol": "protocol",
    "Timestamp": "ts_raw",
    "Flow Duration": "duration_us",
    "Total Fwd Packets": "fwd_pkts",
    "Total Backward Packets": "bwd_pkts",
    "Total Length of Fwd Packets": "fwd_bytes",
    "Total Length of Bwd Packets": "bwd_bytes",
    "Flow IAT Mean": "iat_mean_us",
    "Flow IAT Std": "iat_std_us",
    "SYN Flag Count": "syn_count",
    "RST Flag Count": "rst_count",
    "ACK Flag Count": "ack_count",
    "Init_Win_bytes_forward": "init_win_fwd",
}
LABEL_COLUMN = "attack_label"

# The capture ran roughly 09:00-17:00. Timestamps use a 12-hour clock with no AM/PM marker,
# so hours 1..7 are the afternoon (13..19). This is documented in docs/data-provenance.md.
_PM_BELOW_HOUR = 8


def parse_timestamps(raw: pd.Series) -> pd.Series:
    """Parse CIC-IDS2017 timestamps.

    Formats are mixed ('03/07/2017 08:55:58' on Monday, '5/7/2017 10:58' on later days), are
    day-first, and use a 12-hour clock without AM/PM. Returns naive local capture time.
    """
    ts = pd.to_datetime(raw, dayfirst=True, format="mixed", errors="raise")
    pm = ts.dt.hour < _PM_BELOW_HOUR
    return ts.where(~pm, ts + pd.Timedelta(hours=12))


@dataclass(frozen=True)
class CicData:
    flows: pd.DataFrame          # canonical columns + `ts`; NO label column; sorted by ts; RangeIndex
    labels: pd.Series            # same index as `flows`; dataset authors' attack_label

    def __len__(self) -> int:
        return len(self.flows)

    def select(self, start: pd.Timestamp, end: pd.Timestamp) -> CicData:
        """Time slice [start, end). Index is reset so downstream code can't rely on global row ids."""
        m = (self.flows["ts"] >= start) & (self.flows["ts"] < end)
        flows = self.flows.loc[m].reset_index(drop=True)
        labels = self.labels.loc[m].reset_index(drop=True)
        return CicData(flows, labels)

    def keep_labels(self, labels: Iterable[str]) -> CicData:
        m = self.labels.isin(set(labels))
        return CicData(self.flows.loc[m].reset_index(drop=True), self.labels.loc[m].reset_index(drop=True))


def load_cic(path: str | Path, columns: dict[str, str] | None = None) -> CicData:
    cmap = columns or COLUMN_MAP
    df = pd.read_parquet(path, columns=[*cmap.keys(), LABEL_COLUMN])
    df = df.rename(columns=cmap)
    df["ts"] = parse_timestamps(df["ts_raw"])
    df = df.drop(columns=["ts_raw"]).sort_values("ts", kind="stable").reset_index(drop=True)
    labels = df[LABEL_COLUMN].astype("category")
    flows = df.drop(columns=[LABEL_COLUMN])
    return CicData(flows=flows, labels=labels)


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/demo.py
"""Notebook-friendly helpers: one call to set everything up, one to run a few windows end to end."""
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any

import pandas as pd

from . import llm as llmmod
from .attack_kb import AttackKB, load_chunks
from .baselines import known_destinations
from .config import load_config
from .crew import DownstreamCrew, incident_profile
from .data_cic import CicData, load_cic
from .detector import WindowResult, merge_into_incidents, run_detector
from .pipeline import IncidentOutcome, process_incident
from .profile import BaselineProfile
from .replay import Replay
from .response import Approver, Policy, SimulatedEnvironment, auto_deny
from .schema import Incident


@dataclass
class Demo:
    cfg: dict
    data: CicData
    kb: AttackKB
    profile: BaselineProfile
    known: set[str]
    policy: Policy
    crew: DownstreamCrew
    llm: Any
    mw: llmmod.Middleware


def setup(config_path: str | None = None, model: str | None = None, use_colab_secret: bool = True) -> Demo:
    """Load config, data, knowledge base and agents. On Colab the key comes from Colab Secrets."""
    llmmod.load_dotenv()                                # local runs: a .env file (never overrides the environment)
    if use_colab_secret and not os.environ.get("MISTRAL_API_KEY"):
        try:
            from google.colab import userdata
            os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
        except Exception:                               # noqa: BLE001 - not on Colab / secret missing
            pass
    ok, msg = llmmod.check_api_key()
    print("API key check:", msg)
    if not ok:
        raise RuntimeError(f"Cannot continue: {msg}")
    cfg = load_config(config_path)
    c = cfg["llm"]
    mw = llmmod.install(c["min_interval_s"], c["tokens_per_minute"], c["max_retries"], c["cache_dir"])
    llm = llmmod.make_llm(cfg, model)
    data = load_cic(cfg["data"]["cic_parquet"])
    b = cfg["splits"]["baseline_benign"]
    base = data.select(pd.Timestamp(b["start"]), pd.Timestamp(b["end"])).flows
    kb = AttackKB(load_chunks(cfg["data"]["attack_stix"]), cfg["rag"]["embedding_model"], cfg["rag"]["persist_dir"])
    policy = Policy.from_file(cfg["data"]["policy"])
    return Demo(cfg, data, kb, BaselineProfile(base, cfg["replay"]["window_minutes"]), known_destinations(base),
                policy, DownstreamCrew(kb, policy, llm), llm, mw)


def slice_for(d: Demo, split: str, name: str) -> tuple[dict, CicData]:
    s = next(x for x in d.cfg["splits"][split] if x["name"] == name)
    return s, d.data.select(pd.Timestamp(s["start"]), pd.Timestamp(s["end"]))


def detect(d: Demo, split: str, name: str, windows: list[int], mode: str = "rules") -> tuple[list[WindowResult], list[Incident]]:
    """Run the detector on selected windows. mode: 'rules' (hybrid) or 'screen' (agent-led)."""
    s, sl = slice_for(d, split, name)
    path = f".cache/runs/demo_{mode}_{split}_{name}.jsonl"
    res = run_detector(sl.flows, s["start"], s["end"], d.cfg, d.kb, d.profile, d.llm, path, d.mw,
                       only_windows=windows, candidates_from=mode, known_dsts=d.known)
    findings = [f for r in res for f in r.findings]
    return res, merge_into_incidents(findings, d.cfg["replay"]["window_minutes"])


def respond(d: Demo, incidents: list[Incident], approver: Approver = auto_deny, flows: Any = None
            ) -> tuple[list[IncidentOutcome], SimulatedEnvironment]:
    env = SimulatedEnvironment()
    prof = (lambda i: incident_profile(flows, i)) if flows is not None else (lambda i: "")
    return [process_incident(i, d.crew, d.policy, env, approver, profile=prof(i)) for i in incidents], env


def windows_labelled(d: Demo, split: str, name: str) -> pd.DataFrame:
    """For choosing demo windows: which windows contain attack-labelled flows (grader view, not for agents)."""
    from .evaluate import attack_truth

    s, sl = slice_for(d, split, name)
    rp = Replay(sl.flows, s["start"], s["end"], d.cfg["replay"]["window_minutes"], d.cfg["replay"]["history_windows"])
    return attack_truth(sl.flows, sl.labels, rp.windows(), d.cfg["labels"]["to_technique"],
                        d.cfg["grading"]["min_attack_flows"])


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/detector.py
"""Detector agent: decides what is suspicious in one replay window.

The agent (CrewAI + Mistral) sees a statistics-only digest, can query flows and search ATT&CK, and
answers with JSON findings. Code then verifies every finding (evidence.py); only verified findings leave
this module. Nothing here uses detection rules or labels.
"""
from __future__ import annotations

import json
import os
import re
from collections.abc import Iterable
from dataclasses import dataclass, field
from datetime import timedelta
from pathlib import Path
from typing import Any

os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

import pandas as pd  # noqa: E402

from .attack_kb import AttackKB  # noqa: E402
from .detector_tools import DetectorContext, list_sources, search_attack_kb, source_profile  # noqa: E402
from .digest import entity_stats  # noqa: E402
from .evidence import verify_finding  # noqa: E402
from .llm import Middleware  # noqa: E402
from .profile import BaselineProfile  # noqa: E402
from .replay import Replay, Window  # noqa: E402
from .schema import Finding, Incident  # noqa: E402
from .syncutil import run_blocking  # noqa: E402

SCREEN_TEMPLATE = """You are screening network sources seen in ONE {minutes}-minute window ({label}).
{peers}
For EACH source below decide whether it is doing something its peers are not doing.
Look at: how many destinations and ports it uses, how concentrated its traffic is on one port or one
destination (top_port_share, top_dst_share), how regular its timing is (per_min_cv), and how much data
moves. Ordinary clients and servers are the majority; mark a source unusual only if its pattern differs
clearly from its peers. Volume alone is not enough. Mark at most 3 sources in this list as unusual:
the ones that deviate most from their peers.

Return ONLY this JSON, one entry per source, same order:
{{"sources": [{{"entity": "<ip>", "unusual": false, "why": "<=15 words"}}]}}

SOURCES
{rows}
"""

VERIFY_TEMPLATE = """These sources were flagged as unusual in ONE {minutes}-minute window ({label}).
Decide which of them really behave like an attack or a compromised host.

{peers}
CANDIDATES
{rows}

Method:
1. For each candidate call source_profile and compare it with its baseline and recent history.
2. Describe what it is doing in plain terms, then call search_attack_kb with that description. Confirm a
   candidate only if a retrieved ATT&CK technique genuinely matches the behaviour. Volume alone is not
   an attack, and a destination never seen in the baseline is common for ordinary browsing.
3. Only quote numbers that appear in the rows above or in tool outputs. Only cite technique IDs returned
   by search_attack_kb.
4. You MUST give a verdict for EVERY candidate listed, including the ones you reject.

Allowed evidence metric names: flows, dst_ips, dst_ports, ext_dst_ips, fwd_bytes, bwd_bytes, syn, rst,
active_minutes, per_minute_mean, per_minute_cv, mean_duration_ms, top_port_share, top_dst_share.

Final answer must be ONLY this JSON (no prose), one entry per candidate:
{{"verdicts": [{{"entity": "<source ip>", "attack": true, "behaviour": "<=25 words",
"evidence": {{"flows": 0}}, "technique_hints": [{{"id": "T0000", "citation": "T0000"}}],
"confidence": 0.0}}]}}
Keep it short: at most 4 evidence metrics per candidate. For a rejected candidate use "attack": false,
a behaviour note of at most 8 words, and "evidence": {{}}.
"""

MAX_CANDIDATES = 8
SCREEN_CHUNK = 12


@dataclass
class WindowResult:
    window: Window
    findings: list[Finding] = field(default_factory=list)
    rejected: list[dict[str, Any]] = field(default_factory=list)
    dropped_hints: int = 0
    parse_error: bool = False
    raw: str = ""
    usage: dict[str, float] = field(default_factory=dict)
    screened_unusual: list[str] = field(default_factory=list)
    candidates_truncated: int = 0
    verdicts_missing: int = 0

    def to_json(self) -> dict[str, Any]:
        return {"index": self.window.index, "start": str(self.window.start), "end": str(self.window.end),
                "findings": [f.model_dump(mode="json") for f in self.findings], "rejected": self.rejected,
                "dropped_hints": self.dropped_hints, "parse_error": self.parse_error,
                "raw": self.raw[:2000], "usage": self.usage,
                "screened_unusual": self.screened_unusual, "candidates_truncated": self.candidates_truncated,
                "verdicts_missing": self.verdicts_missing}


def _salvage_objects(text: str) -> list[dict]:
    """Recover complete JSON objects from a truncated answer by balanced-brace scanning (string-aware)."""
    out: list[dict] = []
    start = max(text.find('"verdicts"'), text.find('"findings"'))
    if start < 0:
        return out
    k = text.find("[", start)
    if k < 0:
        return out
    depth, in_str, esc, obj_start = 0, False, False, -1
    for pos in range(k + 1, len(text)):
        ch = text[pos]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            if depth == 0:
                obj_start = pos
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and obj_start >= 0:
                try:
                    obj = json.loads(text[obj_start:pos + 1])
                    if isinstance(obj, dict):
                        out.append(obj)
                except json.JSONDecodeError:
                    pass
                obj_start = -1
            elif depth < 0:
                break
    return out


def parse_findings_json(text: str) -> list[dict] | None:
    """Extract confirmed findings from model output. None means unparseable (nothing salvageable)."""
    if not text:
        return None
    i, j = text.find("{"), text.rfind("}")
    data: Any = None
    if i >= 0 and j > i:
        try:
            data = json.loads(text[i:j + 1])
        except json.JSONDecodeError:
            data = None
    if data is None:                                   # truncated or malformed: keep complete objects
        items = _salvage_objects(text)
        if not items:
            return None
        return [x for x in items if x.get("attack") is True or ("verdicts" not in text and "entity" in x)]
    if not isinstance(data, dict):
        return None
    if isinstance(data.get("verdicts"), list):
        return [x for x in data["verdicts"] if isinstance(x, dict) and x.get("attack") is True]
    legacy = data.get("findings")
    if not isinstance(legacy, list):
        return None
    return [x for x in legacy if isinstance(x, dict)]


def build_tools(ctx: DetectorContext) -> list[Any]:
    from crewai.tools import tool

    @tool("list_sources")
    def list_sources_tool(by: str = "flows", n: int = 8) -> str:
        """Rank ALL sources in the current window by one metric: flows, dst_ports, dst_ips, fwd_bytes or
        regularity (lowest per-minute variation). Use it to look beyond the digest."""
        return list_sources(ctx, by, n)

    @tool("source_profile")
    def source_profile_tool(ip: str) -> str:
        """Detail for one source address: this window, recent history, top destinations (flagging
        never-seen ones) and its normal baseline."""
        return source_profile(ctx, ip)

    @tool("search_attack_kb")
    def search_attack_kb_tool(query: str) -> str:
        """Search the MITRE ATT&CK knowledge base by meaning, e.g. 'many failed logins to one service'.
        Returns technique IDs you may cite."""
        return search_attack_kb(ctx, query)

    return [list_sources_tool, source_profile_tool, search_attack_kb_tool]


def _run_agent(llm: Any, tools: list[Any], prompt: str, max_iter: int) -> str:
    from crewai import Agent, Crew, Process, Task

    agent = Agent(
        role="Network threat detector",
        goal="Find sources whose behaviour in this window indicates an attack, and avoid false alarms.",
        backstory="A careful SOC analyst who verifies suspicions against baselines and ATT&CK before flagging.",
        tools=tools, llm=llm, max_iter=max_iter, allow_delegation=False, verbose=False,
    )
    task = Task(description=prompt, expected_output="Only the JSON object described in the task.", agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False)
    out = run_blocking(crew.kickoff)
    return str(getattr(out, "raw", out))


def parse_screen_json(text: str, allowed: set[str]) -> dict[str, str] | None:
    """Return {entity: why} for sources marked unusual; None if unparseable. Unknown entities are ignored."""
    if not text:
        return None
    i, k = text.find("{"), text.rfind("}")
    if i < 0 or k <= i:
        return None
    try:
        data = json.loads(text[i:k + 1])
    except json.JSONDecodeError:
        return None
    items = data.get("sources") if isinstance(data, dict) else None
    if not isinstance(items, list):
        return None
    out: dict[str, str] = {}
    for x in items:
        if isinstance(x, dict) and x.get("unusual") is True and str(x.get("entity")) in allowed:
            out[str(x["entity"])] = str(x.get("why", ""))[:120]
    return out


def screen_sources(ctx: DetectorContext, llm: Any, minutes: int) -> tuple[dict[str, str], bool]:
    """Stage 1: the model judges EVERY listed source against its peers (forced coverage, small chunks)."""
    ents = ctx.digest.entities
    peers = ("PEERS(in-network median) " + " ".join(f"{k}={v:g}" for k, v in ctx.digest.peers.items())
             if ctx.digest.peers else "")
    flagged: dict[str, str] = {}
    failed = False
    for i in range(0, len(ents), SCREEN_CHUNK):
        part = ents[i:i + SCREEN_CHUNK]
        prompt = SCREEN_TEMPLATE.format(minutes=minutes, label=ctx.window.label(), peers=peers,
                                        rows="\n".join(e.row() for e in part))
        text = str(llm.call([{"role": "system", "content": "You are a careful network security analyst."},
                             {"role": "user", "content": prompt}]))
        got = parse_screen_json(text, {e.entity for e in part})
        if got is None:
            failed = True
        else:
            flagged.update(got)
    return flagged, failed


def rule_candidates(ctx: DetectorContext, rules: dict, known_dsts: set[str] | None) -> dict[str, str]:
    """Hybrid mode: the B0 rules propose candidate sources; the agent verifies them (ADR-019 fallback)."""
    from .baselines import b0_findings

    out: dict[str, str] = {}
    for f in b0_findings(ctx.view.current(), ctx.window, rules, known_dsts):
        out.setdefault(f.entity, f"rule fired: {f.behaviour}")
    return out


def detect_window(ctx: DetectorContext, llm: Any, known_ids: set[str], minutes: int, max_iter: int = 6,
                  mw: Middleware | None = None, candidates_from: str = "screen",
                  rules: dict | None = None, known_dsts: set[str] | None = None) -> WindowResult:
    before = mw.stats.snapshot() if mw else {}
    ctx.trace_start = len(ctx.kb.trace)
    res = WindowResult(window=ctx.window)
    if ctx.digest.total_flows == 0:
        return res

    if candidates_from == "rules":
        flagged, failed = rule_candidates(ctx, rules or {}, known_dsts), False
    else:
        flagged, failed = screen_sources(ctx, llm, minutes)
    res.screened_unusual = sorted(flagged)
    res.parse_error = failed
    if flagged:
        cand = sorted(flagged)[:MAX_CANDIDATES]
        res.candidates_truncated = len(flagged) - len(cand)
        by_ent = {e.entity: e for e in ctx.digest.entities}
        for c in cand:
            if c not in by_ent:                      # a rule fired on a source the digest did not list
                st = entity_stats(ctx.view.current(), c, ctx.window)
                if st is not None:
                    by_ent[c] = st
        peers = ("PEERS(in-network median) " + " ".join(f"{k}={v:g}" for k, v in ctx.digest.peers.items())
                 if ctx.digest.peers else "")
        rows = "\n".join(f"{by_ent[c].row()}  | screening note: {flagged[c]}" for c in cand if c in by_ent)
        prompt = VERIFY_TEMPLATE.format(minutes=minutes, label=ctx.window.label(), peers=peers, rows=rows)
        raw = _run_agent(llm, build_tools(ctx), prompt, max_iter)
        res.raw = raw
        res.verdicts_missing = max(0, len(cand) - len(re.findall(r'"entity"', raw)))
        items = parse_findings_json(raw)
        if items is None:
            res.parse_error = True
        else:
            for draft in items[:MAX_CANDIDATES]:
                v = verify_finding(draft, ctx, known_ids)
                res.dropped_hints += len(v.dropped_hints)
                if v.ok and v.finding is not None:
                    res.findings.append(v.finding)
                else:
                    res.rejected.append({"entity": str(draft.get("entity", "")), "reasons": v.reasons})
    if mw:
        res.usage = mw.stats.delta(before)
    return res


def run_detector(flows: pd.DataFrame, start: str, end: str, cfg: dict, kb: AttackKB, profile: BaselineProfile,
                 llm: Any, out_path: str | Path, mw: Middleware | None = None,
                 only_windows: Iterable[int] | None = None, limit: int | None = None,
                 candidates_from: str = "screen", known_dsts: set[str] | None = None) -> list[WindowResult]:
    """Replay [start, end) window by window. Results are appended to `out_path` (jsonl) so a run resumes."""
    from .digest import build_digest

    rp = Replay(flows, start, end, cfg["replay"]["window_minutes"], cfg["replay"]["history_windows"])
    known = set(kb.chunks)
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    done: dict[int, dict] = {}
    if out.is_file():
        for line in out.read_text(encoding="utf-8").splitlines():
            if line.strip():
                d = json.loads(line)
                done[d["index"]] = d
    wanted = set(only_windows) if only_windows is not None else None
    results: list[WindowResult] = []
    processed = 0
    for w, view in rp:
        if wanted is not None and w.index not in wanted:
            continue
        if w.index in done:
            d = done[w.index]
            results.append(WindowResult(window=w, findings=[Finding(**f) for f in d["findings"]],
                                        rejected=d["rejected"], dropped_hints=d["dropped_hints"],
                                        parse_error=d["parse_error"], raw=d.get("raw", ""), usage=d["usage"],
                                        screened_unusual=d.get("screened_unusual", []),
                                        candidates_truncated=d.get("candidates_truncated", 0),
                                        verdicts_missing=d.get("verdicts_missing", 0)))
            continue
        if limit is not None and processed >= limit:
            break
        dg = build_digest(view.current(), w, cfg["digest"]["max_private"], cfg["digest"]["public_top_k"])
        ctx = DetectorContext(window=w, view=view, digest=dg, kb=kb, profile=profile)
        r = detect_window(ctx, llm, known, cfg["replay"]["window_minutes"], mw=mw,
                          candidates_from=candidates_from, rules=cfg.get("baseline_rules"), known_dsts=known_dsts)
        with out.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r.to_json()) + "\n")
        results.append(r)
        processed += 1
    return results


def merge_into_incidents(findings: Iterable[Finding], window_minutes: int, max_gap_windows: int = 1) -> list[Incident]:
    """Bookkeeping, not detection: join consecutive findings on the same entity into one incident."""
    gap = timedelta(minutes=window_minutes * (max_gap_windows + 1))
    by_entity: dict[str, list[Finding]] = {}
    for f in sorted(findings, key=lambda x: (x.entity, x.window_start)):
        by_entity.setdefault(f.entity, []).append(f)
    out: list[Incident] = []
    for ent, fs in by_entity.items():
        cur: list[Finding] = []
        for f in fs:
            if cur and f.window_start - cur[-1].window_start > gap:
                out.append(_incident(ent, cur, len(out)))
                cur = []
            cur.append(f)
        if cur:
            out.append(_incident(ent, cur, len(out)))
    return sorted(out, key=lambda i: i.start)


def _incident(entity: str, fs: list[Finding], n: int) -> Incident:
    return Incident(id=f"INC-{n + 1:03d}", entity=entity, start=fs[0].window_start, end=fs[-1].window_end,
                    findings=fs)


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/detector_tools.py
"""Read-only tools the detector agent may call. All data flows through the replay `FlowView`, so a tool
can look back a bounded number of windows and never forward, and nothing here can see a label."""
from __future__ import annotations

from dataclasses import dataclass
from typing import cast

import pandas as pd

from .attack_kb import AttackKB
from .digest import build_digest, entity_stats, is_private
from .profile import BaselineProfile
from .replay import FlowView, Window
from .schema import WindowDigest

RANK_KEYS = {"flows", "dst_ports", "dst_ips", "fwd_bytes", "regularity"}


@dataclass
class DetectorContext:
    window: Window
    view: FlowView
    digest: WindowDigest
    kb: AttackKB
    profile: BaselineProfile
    trace_start: int = 0             # index into kb.trace where this window's searches begin

    def retrieved_ids(self) -> set[str]:
        return {i for t in self.kb.trace[self.trace_start:] for i in t["ids"]}


def list_sources(ctx: DetectorContext, by: str = "flows", n: int = 8) -> str:
    """Rank ALL sources in the current window by one neutral metric (not limited to the digest)."""
    if by not in RANK_KEYS:
        return f"unknown ranking '{by}'; choose one of {sorted(RANK_KEYS)}"
    n = max(1, min(int(n), 15))
    cur = ctx.view.current()
    if cur.empty:
        return "no flows in this window"
    g = cur.groupby("src_ip")
    s: pd.Series
    if by == "flows":
        s = g.size()
    elif by == "dst_ports":
        s = g["dst_port"].nunique()
    elif by == "dst_ips":
        s = g["dst_ip"].nunique()
    elif by == "fwd_bytes":
        s = g["fwd_bytes"].sum()
    else:   # regularity: lowest coefficient of variation of flows per minute among sources active >= 3 minutes
        pm = cur.assign(_m=cur["ts"].dt.floor("min")).groupby(["src_ip", "_m"]).size().groupby(level=0)
        cv = (pm.std(ddof=0) / pm.mean()).where(pm.size() >= 3).dropna()
        s = cv.sort_values().head(n)
        return "\n".join(f"{ip} per_min_cv={v:.2f}" for ip, v in s.items()) or "no source active >= 3 minutes"
    top = s.sort_values(ascending=False, kind="stable").head(n)
    sizes = g.size().to_dict()
    return "\n".join(f"{ip} {by}={int(v)} flows={int(sizes[ip])}" for ip, v in top.items())


def source_profile(ctx: DetectorContext, ip: str) -> str:
    """Detail for one source: this window, recent history, top destinations, and its normal baseline."""
    cur = ctx.view.current()
    st = entity_stats(cur, ip, ctx.window)
    if st is None:
        return f"{ip}: no flows as a source in this window"
    out = [f"{ip} private={int(st.is_private)} flows={st.flows} dst_ips={st.dst_ips} dst_ports={st.dst_ports} "
           f"top_ports={','.join(f'{p}:{n}' for p, n in st.top_ports)} ext_dst={st.ext_dst_ips} "
           f"fwd_B={st.fwd_bytes} bwd_B={st.bwd_bytes} syn={st.syn} rst={st.rst} "
           f"act_min={st.active_minutes} per_min={st.per_minute_mean} per_min_cv={st.per_minute_cv}"]
    hist = ctx.view.history()
    h = hist[hist["src_ip"] == ip]
    if not h.empty:
        step_s = (ctx.window.end - ctx.window.start).total_seconds()
        secs = (ctx.window.start - h["ts"]).dt.total_seconds()
        ago = ((secs - 1e-9) // step_s + 1).astype(int).value_counts().sort_index()
        out.append("history flows by windows-ago: " + ", ".join(f"{int(k)}:{int(v)}" for k, v in ago.head(6).items()))
    else:
        out.append("history: no flows from this source in the look-back")
    dest = (cur[cur["src_ip"] == ip].groupby(["dst_ip", "dst_port"]).size().reset_index(name="n")
            .sort_values("n", ascending=False, kind="stable").head(4))
    parts = []
    for r in dest.itertuples(index=False):
        d = str(r.dst_ip)
        tag = "" if is_private(d) else (" (known)" if ctx.profile.destination_known(d) else " (never seen in baseline)")
        parts.append(f"{d}:{int(cast(int, r.dst_port))} x{int(cast(int, r.n))}{tag}")
    out.append("top destinations: " + "; ".join(parts))
    out.append("baseline: " + ", ".join(f"{k}={v}" for k, v in ctx.profile.lookup(ip).items()))
    return "\n".join(out)


def search_attack_kb(ctx: DetectorContext, query: str) -> str:
    """Search the official MITRE ATT&CK knowledge base by meaning. Returns IDs you may cite."""
    hits = ctx.kb.search(query, k=4)
    return "\n".join(f"{h.id} {h.name} [{', '.join(h.tactics)}] score={h.score:.2f}: {h.snippet[:170]}" for h in hits)


def window_digest_text(flows: pd.DataFrame, window: Window, max_private: int = 40, public_top_k: int = 6) -> str:
    return build_digest(flows, window, max_private, public_top_k).to_text()


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/digest.py
"""Window digest: per-source statistics computed with arithmetic only.

Deliberately contains no thresholds and no verdicts, so the decision of what is suspicious stays
with the detector agent.

A window can hold hundreds of sources (in CIC-IDS2017 ~13 internal hosts and ~500 public addresses with
1-3 flows each), so the listing is structured, not truncated by one ranking:
  * every private (in-network) source is listed;
  * public sources are listed if they lead in ANY of several neutral views (flows, destinations,
    ports, outbound bytes, most regular timing), so a quiet source is not hidden by volume alone;
  * everything else is summarised with explicit counts, never silently dropped.
The agent's tools can query any source individually, so the digest is a starting point.
"""
from __future__ import annotations

import ipaddress
from functools import cache

import numpy as np
import pandas as pd

from .replay import Window
from .schema import EntityStats, WindowDigest

_RFC1918 = [ipaddress.ip_network(n) for n in ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]


@cache
def is_private(ip: str) -> bool:
    """RFC 1918 address (an in-network host). Deliberately narrower than ipaddress.is_private,
    which also covers loopback, link-local and documentation ranges."""
    try:
        addr = ipaddress.ip_address(ip)
    except ValueError:
        return False
    return any(addr in net for net in _RFC1918)


def _ip_key(ip: str) -> tuple[int, int]:
    try:
        a = ipaddress.ip_address(ip)
        return (a.version, int(a))
    except ValueError:
        return (0, 0)


def build_digest(flows: pd.DataFrame, window: Window, max_private: int = 40,
                 public_top_k: int = 6) -> WindowDigest:
    if flows.empty:
        return WindowDigest(window_start=window.start, window_end=window.end, total_flows=0,
                            sources=0, entities=[])

    df = flows
    minute = df["ts"].dt.floor("min")
    g = df.groupby("src_ip", sort=False)
    base = pd.DataFrame({
        "flows": g.size(),
        "dst_ips": g["dst_ip"].nunique(),
        "dst_ports": g["dst_port"].nunique(),
        "mean_duration_ms": g["duration_us"].mean() / 1000.0,
        "fwd_bytes": g["fwd_bytes"].sum(),
        "bwd_bytes": g["bwd_bytes"].sum(),
        "syn": g["syn_count"].sum(),
        "rst": g["rst_count"].sum(),
    })

    ext = df.loc[~df["dst_ip"].map(is_private), ["src_ip", "dst_ip"]]
    base["ext_dst_ips"] = ext.groupby("src_ip")["dst_ip"].nunique().reindex(base.index).fillna(0).astype(int)

    per_min = df.assign(_m=minute).groupby(["src_ip", "_m"]).size()
    pm = per_min.groupby(level=0)
    base["active_minutes"] = pm.size()
    base["per_minute_mean"] = pm.mean()
    std = pm.std(ddof=0)
    base["per_minute_cv"] = (std / base["per_minute_mean"]).where(base["active_minutes"] >= 2, 0.0).fillna(0.0)

    ports = df.groupby(["src_ip", "dst_port"]).size().rename("n").reset_index()
    ports = ports.sort_values(["src_ip", "n", "dst_port"], ascending=[True, False, True])
    port_max = ports.groupby("src_ip")["n"].max()
    dst_max = df.groupby(["src_ip", "dst_ip"]).size().groupby(level=0).max()
    base["top_port_share"] = (port_max / base["flows"]).reindex(base.index).fillna(0.0)
    base["top_dst_share"] = (dst_max / base["flows"]).reindex(base.index).fillna(0.0)
    top: dict[str, list[tuple[int, int]]] = {}
    for src, port, n in ports.groupby("src_ip").head(3).itertuples(index=False):
        top.setdefault(str(src), []).append((int(port), int(n)))

    base = base.sort_values("flows", ascending=False, kind="stable")
    private = base[[is_private(str(i)) for i in base.index]]
    public = base.drop(index=private.index)
    chosen = list(private.head(max_private).index)
    views = [public.nlargest(public_top_k, col, keep="first").index
             for col in ("flows", "dst_ports", "dst_ips", "fwd_bytes")]
    regular = public[public["active_minutes"] >= 3].nsmallest(public_top_k, "per_minute_cv", keep="first").index
    for idx in (*views, regular):
        chosen.extend(i for i in idx if i not in chosen)
    # neutral order (numeric by address), not by volume, so row position carries no signal
    shown = base.loc[sorted(chosen, key=lambda i: _ip_key(str(i)))]
    rest = base.drop(index=shown.index)
    peers: dict[str, float] = {}
    if len(private):
        peers = {"flows": float(private["flows"].median()), "dst_ips": float(private["dst_ips"].median()),
                 "dst_ports": float(private["dst_ports"].median()),
                 "per_min_cv": float(np.round(private["per_minute_cv"].median(), 2))}
    entities = [
        EntityStats(
            entity=str(ip), is_private=is_private(str(ip)), flows=int(r.flows), dst_ips=int(r.dst_ips),
            dst_ports=int(r.dst_ports), top_ports=top.get(str(ip), []), ext_dst_ips=int(r.ext_dst_ips),
            mean_duration_ms=float(np.round(r.mean_duration_ms, 3)), fwd_bytes=int(r.fwd_bytes),
            bwd_bytes=int(r.bwd_bytes), syn=int(r.syn), rst=int(r.rst), active_minutes=int(r.active_minutes),
            per_minute_mean=float(np.round(r.per_minute_mean, 3)), per_minute_cv=float(np.round(r.per_minute_cv, 3)),
            top_port_share=float(np.round(r.top_port_share, 3)), top_dst_share=float(np.round(r.top_dst_share, 3)),
        )
        for ip, r in shown.iterrows()
    ]
    return WindowDigest(window_start=window.start, window_end=window.end, total_flows=int(len(df)),
                        sources=int(len(base)), entities=entities, omitted=int(len(rest)),
                        omitted_flows=int(rest["flows"].sum()) if len(rest) else 0,
                        omitted_max_flows=int(rest["flows"].max()) if len(rest) else 0,
                        omitted_max_dst_ports=int(rest["dst_ports"].max()) if len(rest) else 0,
                        peers=peers)


def entity_stats(flows: pd.DataFrame, ip: str, window: Window) -> EntityStats | None:
    """Statistics for one source in `flows` (same arithmetic as the digest), or None if it is absent."""
    sub = flows[flows["src_ip"] == ip]
    if sub.empty:
        return None
    return build_digest(sub, window, max_private=1, public_top_k=1).entities[0]


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/evaluate.py
"""Grading by plain code against the dataset's own labels. Nothing here calls an LLM.

Granularity is (window, entity): a source in a window is attack-positive when it has at least
`min_attack_flows` attack-labelled flows in that window (labels are not entity-clean, see
docs/data-provenance.md). Technique matching is parent-level (T1110 accepts T1110.001).
"""
from __future__ import annotations

from collections import defaultdict
from collections.abc import Iterable
from typing import cast

import pandas as pd

from .replay import Window
from .schema import Finding


def attack_truth(flows: pd.DataFrame, labels: pd.Series, windows: list[Window],
                 label_to_technique: dict[str, str], min_attack_flows: int) -> pd.DataFrame:
    """Rows: window_index, entity, label, technique, n_attack_flows. Attack labels only."""
    if not windows:
        return pd.DataFrame(columns=["window_index", "entity", "label", "technique", "n_attack_flows"])
    start, step = windows[0].start, windows[0].end - windows[0].start
    lab = labels.astype(str).reset_index(drop=True)
    f = flows.reset_index(drop=True)
    mask = lab.isin(label_to_technique)
    df = pd.DataFrame({"entity": f.loc[mask, "src_ip"].values, "label": lab[mask].values,
                       "idx": ((f.loc[mask, "ts"] - start) // step).astype(int).values})
    df = df[(df.idx >= 0) & (df.idx < len(windows))]
    g = df.groupby(["idx", "entity", "label"]).size().rename("n_attack_flows").reset_index()
    g = g[g.n_attack_flows >= min_attack_flows].rename(columns={"idx": "window_index"})
    g["technique"] = g["label"].map(label_to_technique)
    return g.reset_index(drop=True)


def parent(technique_id: str) -> str:
    return technique_id.split(".")[0]


def score_findings(findings: Iterable[Finding], truth: pd.DataFrame, windows: list[Window]) -> dict:
    """(window, entity)-level detection scores and parent-level technique accuracy on true positives."""
    starts = {pd.Timestamp(w.start): w.index for w in windows}
    pred: dict[tuple[int, str], list[str]] = defaultdict(list)
    for f in findings:
        ws = pd.Timestamp(f.window_start)
        if not f.suspicious or ws not in starts:
            continue
        pred[(starts[ws], f.entity)].extend(h.id for h in f.technique_hints)

    truth_tech: dict[tuple[int, str], set[str]] = defaultdict(set)
    for r in truth.itertuples(index=False):
        truth_tech[(int(cast(int, r.window_index)), str(r.entity))].add(str(r.technique))

    keys_t, keys_p = set(truth_tech), set(pred)
    tp, fp, fn = keys_t & keys_p, keys_p - keys_t, keys_t - keys_p
    precision = len(tp) / (len(tp) + len(fp)) if (tp or fp) else 0.0
    recall = len(tp) / len(keys_t) if keys_t else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    tech_ok = sum(1 for k in tp if {parent(h) for h in pred[k]} & truth_tech[k])
    attack_windows = {k[0] for k in keys_t}
    hit_windows = {k[0] for k in tp}
    per_tech: dict[str, dict[str, int]] = defaultdict(lambda: {"truth": 0, "tp": 0})
    for k, techs in truth_tech.items():
        for t in techs:
            per_tech[t]["truth"] += 1
            per_tech[t]["tp"] += int(k in tp)
    return {
        "truth_positives": len(keys_t), "predicted_positives": len(keys_p),
        "tp": len(tp), "fp": len(fp), "fn": len(fn),
        "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
        "window_recall": round(len(hit_windows) / len(attack_windows), 3) if attack_windows else 0.0,
        "technique_accuracy_on_tp": round(tech_ok / len(tp), 3) if tp else 0.0,
        "per_technique": dict(per_tech),
    }


def grade_outcomes(outcomes: list, truth: pd.DataFrame, windows: list[Window], kb_ids: set[str]) -> dict:
    """Incident-level grading of the full pipeline (technique naming, grounding, response quality).

    An incident is *attack-linked* when at least one of its (window, entity) findings is attack-positive.
    Technique accuracy is measured on attack-linked incidents only, at parent level.
    """
    starts = {pd.Timestamp(w.start): w.index for w in windows}
    truth_tech: dict[tuple[int, str], set[str]] = defaultdict(set)
    for r in truth.itertuples(index=False):
        truth_tech[(int(cast(int, r.window_index)), str(r.entity))].add(str(r.technique))

    linked = correct = 0
    cited = grounded = 0
    actions = executed_or_denied = with_mitigation = 0
    for o in outcomes:
        keys = {(starts.get(pd.Timestamp(f.window_start), -1), f.entity) for f in o.incident.findings}
        want: set[str] = set()
        for k in keys:
            want |= truth_tech.get(k, set())
        if want:
            linked += 1
            got = {parent(t.id) for t in o.assessment.techniques}
            correct += int(bool(got & want))
        n_ok, n_drop = len(o.assessment.techniques), len(o.assessment.dropped_ids)
        cited += n_ok + n_drop
        grounded += sum(1 for t in o.assessment.techniques if t.id in kb_ids)
        for r in o.log.records:
            actions += 1
            executed_or_denied += int(r.status != "rejected")
            with_mitigation += int(r.action.mitigation_id is not None)
        actions += len(o.guard.dropped_actions)               # unknown action types never reach the executor
    return {
        "incidents": len(outcomes), "attack_linked_incidents": linked,
        "incident_precision": round(linked / len(outcomes), 3) if outcomes else 0.0,
        "technique_accuracy_incident": round(correct / linked, 3) if linked else 0.0,
        "grounding_rate": round(grounded / cited, 3) if cited else 1.0,
        "policy_pass_rate": round(executed_or_denied / actions, 3) if actions else 1.0,
        "mitigation_grounded_share": round(with_mitigation / actions, 3) if actions else 0.0,
        "proposed_actions": actions,
    }


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/evidence.py
"""Evidence check: code verifies what the detector claims, before anything downstream trusts it.

A finding survives only if its entity exists in the window and none of its quoted numbers contradicts
the real statistics. Technique hints must be real ATT&CK IDs that were actually retrieved in this window.
"""
from __future__ import annotations

from dataclasses import dataclass, field

from .detector_tools import DetectorContext
from .digest import entity_stats
from .schema import Finding, TechniqueHint

CHECKABLE = ("flows", "dst_ips", "dst_ports", "ext_dst_ips", "fwd_bytes", "bwd_bytes", "syn", "rst",
             "active_minutes", "per_minute_mean", "per_minute_cv", "mean_duration_ms", "top_port_share", "top_dst_share")


@dataclass
class Verdict:
    ok: bool
    reasons: list[str] = field(default_factory=list)
    verified: int = 0
    unverifiable: int = 0
    dropped_hints: list[str] = field(default_factory=list)
    finding: Finding | None = None


def _close(claimed: float, actual: float) -> bool:
    return abs(claimed - actual) <= max(0.05, 0.05 * abs(actual))


def verify_finding(draft: dict, ctx: DetectorContext, known_ids: set[str]) -> Verdict:
    entity = str(draft.get("entity", "")).strip()
    st = entity_stats(ctx.view.current(), entity, ctx.window) if entity else None
    if st is None:
        return Verdict(False, [f"entity {entity or '<empty>'} has no flows in this window"])

    v = Verdict(True)
    actual = st.model_dump()
    clean_ev: dict[str, float] = {}
    for k, raw in (draft.get("evidence") or {}).items():
        try:
            claimed = float(raw)
        except (TypeError, ValueError):
            v.unverifiable += 1
            continue
        if k not in CHECKABLE:
            v.unverifiable += 1
            continue
        if _close(claimed, float(actual[k])):
            v.verified += 1
            clean_ev[k] = claimed
        else:
            v.ok = False
            v.reasons.append(f"{k}: claimed {claimed:g}, actual {float(actual[k]):g}")
    if v.ok and v.verified == 0:
        v.ok = False
        v.reasons.append("no verifiable evidence quoted")

    retrieved = ctx.retrieved_ids()
    hints: list[TechniqueHint] = []
    for h in draft.get("technique_hints") or []:
        tid = str((h or {}).get("id", "")).strip()
        if tid in known_ids and tid in retrieved:
            hints.append(TechniqueHint(id=tid, citation=str((h or {}).get("citation", tid))))
        elif tid:
            v.dropped_hints.append(tid)

    if v.ok:
        try:
            conf = max(0.0, min(1.0, float(draft.get("confidence", 0.5))))
        except (TypeError, ValueError):
            conf = 0.5
        v.finding = Finding(window_start=ctx.window.start, window_end=ctx.window.end, entity=entity,
                            behaviour=str(draft.get("behaviour", ""))[:300], evidence=clean_ev,
                            technique_hints=hints, confidence=conf, suspicious=True)
    return v


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/llm.py
"""LLM plumbing: .env loading, a throttle + response cache around `litellm.completion`, usage stats.

CrewAI routes non-native providers (Mistral) through `litellm.completion`, so wrapping that one function
gives us, without touching CrewAI internals:
  * a request-rate throttle and a tokens-per-minute budget (the free tier is ~20k tokens/min on Small);
  * an on-disk response cache keyed by the full request (re-runs and the demo cost nothing);
  * retries with exponential backoff on rate limits and transient errors;
  * usage counters for the cost metrics.
Caching is at the completion level, so tools still execute on cached runs and retrieval traces stay real.
"""
from __future__ import annotations

import hashlib
import json
import os
import threading
import time
from collections import deque
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

_RETRY_STATUS = {408, 409, 429, 500, 502, 503, 504}


def load_dotenv(path: str | Path = ".env") -> bool:
    """Minimal .env reader (KEY=VALUE). Never overrides variables already set. Returns True if file found."""
    p = Path(path)
    if not p.is_file():
        return False
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    return True


@dataclass
class LLMStats:
    calls: int = 0                 # requests that reached the provider
    cache_hits: int = 0
    retries: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    waited_s: float = 0.0
    log: list[dict] = field(default_factory=list)

    def snapshot(self) -> dict[str, float]:
        return {"calls": self.calls, "cache_hits": self.cache_hits, "retries": self.retries,
                "prompt_tokens": self.prompt_tokens, "completion_tokens": self.completion_tokens,
                "waited_s": round(self.waited_s, 1)}

    def delta(self, before: dict[str, float]) -> dict[str, float]:
        now = self.snapshot()
        return {k: round(now[k] - before.get(k, 0), 1) for k in now}


_UNSUPPORTED_MESSAGE_KEYS = {"cache_breakpoint", "cache_control"}


def sanitize_messages(obj: Any) -> Any:
    """Drop client-side prompt-caching hints that CrewAI adds but the Mistral API rejects."""
    if isinstance(obj, dict):
        return {k: sanitize_messages(v) for k, v in obj.items() if k not in _UNSUPPORTED_MESSAGE_KEYS}
    if isinstance(obj, list):
        return [sanitize_messages(x) for x in obj]
    return obj


def _estimate_tokens(kwargs: dict[str, Any]) -> int:
    text = json.dumps(kwargs.get("messages", []), default=str) + json.dumps(kwargs.get("tools", []), default=str)
    return int(len(text) / 3.2) + int(kwargs.get("max_tokens") or 500)


def _cache_key(kwargs: dict[str, Any]) -> str:
    keep = {k: kwargs.get(k) for k in ("model", "messages", "temperature", "tools", "tool_choice",
                                       "stop", "max_tokens", "response_format", "top_p", "seed")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()


class Middleware:
    def __init__(self, min_interval_s: float = 1.2, tokens_per_minute: int = 16000, max_retries: int = 5,
                 cache_dir: str | Path | None = None, sleep: Callable[[float], None] = time.sleep,
                 clock: Callable[[], float] = time.monotonic):
        self.min_interval_s = min_interval_s
        self.tpm = tokens_per_minute
        self.max_retries = max_retries
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.stats = LLMStats()
        self._sleep, self._clock = sleep, clock
        self._lock = threading.Lock()
        self._last = -1e9
        self._window: deque[tuple[float, int]] = deque()     # (time, tokens) in the last 60 s

    # --- throttle -----------------------------------------------------------------------------
    def _wait_for_capacity(self, need: int) -> None:
        while True:
            with self._lock:
                now = self._clock()
                while self._window and now - self._window[0][0] >= 60:
                    self._window.popleft()
                used = sum(t for _, t in self._window)
                gap = self.min_interval_s - (now - self._last)
                if gap <= 0 and (used + need <= self.tpm or not self._window):
                    self._last = now
                    self._window.append((now, need))
                    return
                if gap > 0:
                    delay = gap
                else:
                    delay = max(60 - (now - self._window[0][0]), 0.5)
            self.stats.waited_s += delay
            self._sleep(delay)

    def _record_actual(self, estimated: int, actual: int) -> None:
        with self._lock:
            for i in range(len(self._window) - 1, -1, -1):
                t, n = self._window[i]
                if n == estimated:
                    self._window[i] = (t, actual)
                    break

    # --- cache ----------------------------------------------------------------------------------
    def _cache_path(self, key: str) -> Path | None:
        return self.cache_dir / key[:2] / f"{key}.json" if self.cache_dir else None

    def _cache_get(self, key: str) -> dict | None:
        p = self._cache_path(key)
        if p and p.is_file():
            try:
                return json.loads(p.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                return None
        return None

    def _cache_put(self, key: str, payload: dict) -> None:
        p = self._cache_path(key)
        if p:
            p.parent.mkdir(parents=True, exist_ok=True)
            tmp = p.with_suffix(".tmp")
            tmp.write_text(json.dumps(payload), encoding="utf-8")
            tmp.replace(p)

    # --- wrapper --------------------------------------------------------------------------------
    def wrap(self, completion: Callable[..., Any], make_response: Callable[[dict], Any]) -> Callable[..., Any]:
        def wrapped(*args: Any, **kwargs: Any) -> Any:
            if args or kwargs.get("stream"):
                return completion(*args, **kwargs)           # streaming/positional: pass through untouched
            if "messages" in kwargs:
                kwargs = {**kwargs, "messages": sanitize_messages(kwargs["messages"])}
            key = _cache_key(kwargs)
            hit = self._cache_get(key)
            if hit is not None:
                self.stats.cache_hits += 1
                return make_response(hit)
            need = _estimate_tokens(kwargs)
            for attempt in range(self.max_retries + 1):
                self._wait_for_capacity(need)
                try:
                    resp = completion(**kwargs)
                    break
                except Exception as e:                       # noqa: BLE001 - classified below
                    status = getattr(e, "status_code", None)
                    transient = status in _RETRY_STATUS or type(e).__name__ in {
                        "RateLimitError", "Timeout", "APIConnectionError", "ServiceUnavailableError",
                        "InternalServerError"}
                    if not transient or attempt == self.max_retries:
                        raise
                    self.stats.retries += 1
                    backoff = min(2 ** (attempt + 1), 60)
                    self.stats.waited_s += backoff
                    self._sleep(backoff)
            usage = getattr(resp, "usage", None)
            pt = int(getattr(usage, "prompt_tokens", 0) or 0)
            ct = int(getattr(usage, "completion_tokens", 0) or 0)
            self.stats.calls += 1
            self.stats.prompt_tokens += pt
            self.stats.completion_tokens += ct
            if pt + ct:
                self._record_actual(need, pt + ct)
            payload = resp.model_dump() if hasattr(resp, "model_dump") else resp
            self._cache_put(key, payload)
            return resp
        return wrapped


_installed: Middleware | None = None


def install(min_interval_s: float = 1.2, tokens_per_minute: int = 16000, max_retries: int = 5,
            cache_dir: str | Path | None = ".cache/llm") -> Middleware:
    """Patch `litellm.completion` once per process. Returns the active middleware (for stats)."""
    global _installed
    if _installed is not None:
        return _installed
    import litellm

    mw = Middleware(min_interval_s, tokens_per_minute, max_retries, cache_dir)
    litellm.completion = mw.wrap(litellm.completion, lambda d: litellm.ModelResponse(**d))
    _installed = mw
    return mw


def check_api_key(timeout: float = 20.0) -> tuple[bool, str]:
    """Preflight: does the configured key authenticate? Never prints or returns the key itself."""
    import urllib.error
    import urllib.request

    key = os.environ.get("MISTRAL_API_KEY", "")
    clean = key.strip().strip('"').strip("'")
    if clean != key:
        os.environ["MISTRAL_API_KEY"] = clean
    if not clean:
        return False, "MISTRAL_API_KEY is not set"
    req = urllib.request.Request("https://api.mistral.ai/v1/models", headers={"Authorization": f"Bearer {clean}"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return True, f"key accepted (length {len(clean)}, HTTP {r.status})"
    except urllib.error.HTTPError as e:
        hint = " - re-copy the key into the Colab secret / .env without spaces" if e.code == 401 else ""
        return False, f"provider rejected the key (length {len(clean)}, HTTP {e.code}){hint}"
    except OSError as e:
        return False, f"could not reach the provider: {e}"


def make_llm(cfg: dict, model: str | None = None):
    """CrewAI LLM for the configured model. Call `install()` first so the middleware is in place."""
    from crewai import LLM

    load_dotenv()
    c = cfg["llm"]
    return LLM(model=model or c["model"], temperature=c.get("temperature", 0.0),
               max_tokens=c.get("max_tokens", 700), max_retries=0)


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/pipeline.py
"""End-to-end orchestration for incidents: assess -> plan -> execute -> report."""
from __future__ import annotations

from dataclasses import dataclass, field

from .crew import DownstreamCrew, PlanGuard
from .response import Approver, Policy, SimulatedEnvironment, auto_deny, execute_plan
from .schema import ActionLog, Incident, ResponsePlan, ThreatAssessment


@dataclass
class IncidentOutcome:
    incident: Incident
    assessment: ThreatAssessment
    plan: ResponsePlan
    guard: PlanGuard
    log: ActionLog
    report: str = ""
    raw: dict[str, str] = field(default_factory=dict)


def log_summary(log: ActionLog) -> str:
    if not log.records:
        return "no actions proposed"
    return "; ".join(f"{r.action.type} {r.action.target} -> {r.status}"
                     + (f" ({r.reason})" if r.status in {"rejected", "denied"} else "") for r in log.records)


def process_incident(inc: Incident, crew: DownstreamCrew, policy: Policy, env: SimulatedEnvironment,
                     approver: Approver = auto_deny, write_report: bool = True,
                     profile: str = "") -> IncidentOutcome:
    assessment, raw_a = crew.assess(inc, profile)
    guard, raw_p = crew.plan(inc, assessment, profile)
    log = execute_plan(guard.plan, policy, env, approver)
    report = crew.report(inc, assessment, guard.plan, log_summary(log)) if write_report else ""
    return IncidentOutcome(inc, assessment, guard.plan, guard, log, report, {"assess": raw_a, "plan": raw_p})


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/profile.py
"""Baseline profile of normal traffic (from the benign-only Monday), used as context for the detector.

This is a lookup table of what is usual per source, not a verdict: the detector decides what to do with it.
"""
from __future__ import annotations

import pandas as pd


class BaselineProfile:
    def __init__(self, flows: pd.DataFrame, window_minutes: int = 5):
        bucket = flows["ts"].dt.floor(f"{window_minutes}min")
        per = (flows.assign(_w=bucket).groupby(["src_ip", "_w"])
               .agg(flows=("dst_ip", "size"), ports=("dst_port", "nunique")))
        g = per.groupby(level=0)
        self.n_windows = int(bucket.nunique())
        self._active = g.size().to_dict()
        self._med = g["flows"].median().to_dict()
        self._p95 = g["flows"].quantile(0.95).to_dict()
        self._max = g["flows"].max().to_dict()
        self._ports_med = g["ports"].median().to_dict()
        self._ports_p95 = g["ports"].quantile(0.95).to_dict()
        self._dsts = set(flows["dst_ip"].astype(str).unique())

    def lookup(self, ip: str) -> dict:
        if ip not in self._active:
            return {"seen_in_baseline": False}
        return {
            "seen_in_baseline": True,
            "active_windows": f"{int(self._active[ip])}/{self.n_windows}",
            "flows_per_window_median": float(self._med[ip]),
            "flows_per_window_p95": float(self._p95[ip]),
            "flows_per_window_max": int(self._max[ip]),
            "dst_ports_per_window_median": float(self._ports_med[ip]),
            "dst_ports_per_window_p95": float(self._ports_p95[ip]),
        }

    def destination_known(self, ip: str) -> bool:
        return ip in self._dsts


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/replay.py
"""Replay simulator: reveals traffic one time window at a time.

Everything an agent tool may read goes through `FlowView`, which is bound to a single window. It can
look back a bounded number of windows and can never look forward. It also holds no label column, so
there is nothing to leak.
"""
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass

import pandas as pd


class FutureAccessError(RuntimeError):
    """A tool tried to read traffic after the end of the current window."""


class HistoryLimitError(RuntimeError):
    """A tool tried to look back further than the configured history."""


@dataclass(frozen=True)
class Window:
    index: int
    start: pd.Timestamp
    end: pd.Timestamp

    def label(self) -> str:
        return f"{self.start:%Y-%m-%d %H:%M}-{self.end:%H:%M}"


class FlowView:
    """Read-only view of flows up to the end of `window`. No labels, no future."""

    def __init__(self, flows: pd.DataFrame, window: Window, step: pd.Timedelta, history_windows: int):
        self._flows = flows
        self.window = window
        self._step = step
        self._history = history_windows

    @property
    def earliest(self) -> pd.Timestamp:
        return self.window.start - self._history * self._step

    def _between(self, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        ts = self._flows["ts"]
        lo = ts.searchsorted(start, side="left")
        hi = ts.searchsorted(end, side="left")
        return self._flows.iloc[lo:hi]

    def current(self) -> pd.DataFrame:
        return self._between(self.window.start, self.window.end)

    def history(self) -> pd.DataFrame:
        """Flows from the allowed look-back up to (not including) the current window."""
        return self._between(self.earliest, self.window.start)

    def between(self, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        if end > self.window.end:
            raise FutureAccessError(f"requested up to {end}, but replay clock is at {self.window.end}")
        if start < self.earliest:
            raise HistoryLimitError(f"requested from {start}, earliest allowed is {self.earliest}")
        return self._between(start, end)


class Replay:
    def __init__(self, flows: pd.DataFrame, start: str | pd.Timestamp, end: str | pd.Timestamp,
                 window_minutes: int = 5, history_windows: int = 12):
        leaked = [c for c in flows.columns if "label" in c.lower()]
        if leaked:
            raise ValueError(f"flows must not carry label columns: {leaked}")
        if not flows["ts"].is_monotonic_increasing:
            raise ValueError("flows must be sorted by ts")
        self._flows = flows
        self.start, self.end = pd.Timestamp(start), pd.Timestamp(end)
        self.step = pd.Timedelta(minutes=window_minutes)
        self.history_windows = history_windows

    def windows(self) -> list[Window]:
        out, t, i = [], self.start, 0
        while t < self.end:
            out.append(Window(i, t, min(t + self.step, self.end)))
            t += self.step
            i += 1
        return out

    def view(self, window: Window) -> FlowView:
        return FlowView(self._flows, window, self.step, self.history_windows)

    def __iter__(self) -> Iterator[tuple[Window, FlowView]]:
        for w in self.windows():
            yield w, self.view(w)


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/response.py
"""Response layer: policy, simulated environment, approval gate and executor. Plain code, no LLM.

Everything a plan proposes is checked here before it touches (the simulated) environment:
  * the action must be in the catalog;
  * targets on the protected-asset list can never be blocked or isolated;
  * `low` risk actions run automatically, `medium`/`high` need an explicit human approval.
"""
from __future__ import annotations

import ipaddress
import json
import re
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from .schema import ActionLog, ActionRecord, ProposedAction, ResponsePlan

IP_ACTIONS = {"block_ip", "rate_limit_ip", "isolate_host"}
_DOMAIN = re.compile(r"^(?=.{1,253}$)([a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}$", re.I)


@dataclass(frozen=True)
class Decision:
    status: str          # auto | needs_approval | rejected
    reason: str
    risk: str = ""


class Policy:
    def __init__(self, catalog: dict[str, dict[str, str]], protected_ips: set[str]):
        self.catalog = catalog
        self.protected = protected_ips

    @classmethod
    def from_file(cls, path: str | Path) -> Policy:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(data["action_catalog"], {a["ip"] for a in data.get("protected_assets", [])})

    def check(self, a: ProposedAction) -> Decision:
        spec = self.catalog.get(a.type)
        if spec is None:
            return Decision("rejected", f"unknown action '{a.type}'")
        risk = spec["risk"]
        if a.type in IP_ACTIONS:
            try:
                ipaddress.ip_address(a.target)
            except ValueError:
                return Decision("rejected", f"target '{a.target}' is not an IP address", risk)
            if a.target in self.protected:
                return Decision("rejected", f"{a.target} is a protected asset", risk)
        if a.type == "block_domain" and not _DOMAIN.match(a.target):
            return Decision("rejected", f"target '{a.target}' is not a domain name", risk)
        if risk == "low":
            return Decision("auto", "low risk", risk)
        return Decision("needs_approval", f"{risk} risk action requires approval", risk)


@dataclass
class SimulatedEnvironment:
    """A pretend firewall/IAM/ticketing system. Nothing here reaches a real network."""

    blocked_ips: set[str] = field(default_factory=set)
    rate_limited: set[str] = field(default_factory=set)
    blocked_domains: set[str] = field(default_factory=set)
    isolated_hosts: set[str] = field(default_factory=set)
    disabled_accounts: set[str] = field(default_factory=set)
    password_resets: set[str] = field(default_factory=set)
    waf_rules: list[str] = field(default_factory=list)
    tickets: list[str] = field(default_factory=list)

    def apply(self, a: ProposedAction) -> None:
        t = a.type
        if t == "block_ip":
            self.blocked_ips.add(a.target)
        elif t == "rate_limit_ip":
            self.rate_limited.add(a.target)
        elif t == "block_domain":
            self.blocked_domains.add(a.target.lower())
        elif t == "isolate_host":
            self.isolated_hosts.add(a.target)
        elif t == "disable_account":
            self.disabled_accounts.add(a.target)
        elif t == "force_password_reset":
            self.password_resets.add(a.target)
        elif t == "enable_waf_rule":
            self.waf_rules.append(a.target)
        elif t == "open_ticket":
            self.tickets.append(f"{a.target}: {a.justification}"[:200])
        else:                                               # policy should have stopped this
            raise ValueError(f"unsupported action {t}")

    def snapshot(self) -> dict[str, Any]:
        return {k: (sorted(v) if isinstance(v, set) else list(v)) for k, v in self.__dict__.items()}


Approver = Callable[[ProposedAction, str], bool]


def auto_deny(action: ProposedAction, reason: str) -> bool:        # safe default for unattended runs
    return False


def auto_approve(action: ProposedAction, reason: str) -> bool:     # for tests and scripted demos only
    return True


def console_approver(action: ProposedAction, reason: str) -> bool:
    """Interactive gate for the notebook: asks the human."""
    print(f"\nAPPROVAL NEEDED: {action.type} -> {action.target}\n  why: {action.justification}\n  {reason}")
    return input("approve? [y/N] ").strip().lower() in {"y", "yes"}


def execute_plan(plan: ResponsePlan, policy: Policy, env: SimulatedEnvironment,
                 approver: Approver = auto_deny) -> ActionLog:
    log = ActionLog(incident_id=plan.incident_id)
    for a in plan.actions:
        d = policy.check(a)
        if d.status == "rejected":
            log.records.append(ActionRecord(action=a, status="rejected", reason=d.reason, risk=d.risk))
        elif d.status == "auto" or approver(a, d.reason):
            env.apply(a)
            log.records.append(ActionRecord(action=a, status="executed", reason=d.reason, risk=d.risk))
        else:
            log.records.append(ActionRecord(action=a, status="denied", reason="approval denied", risk=d.risk))
    return log


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/schema.py
"""Shared data contracts for the agent side of the system. Bulk flow data stays in DataFrames
(see data_cic.py); these models carry what agents and graders exchange."""
from __future__ import annotations

from datetime import datetime

from pydantic import BaseModel, Field


class TechniqueHint(BaseModel):
    id: str                         # ATT&CK ID, e.g. T1110
    citation: str = ""              # KB chunk the model claims to rely on


class Finding(BaseModel):
    """One detector-agent verdict about one entity in one window. Never contains a label."""

    window_start: datetime
    window_end: datetime
    entity: str                     # usually a source IP
    behaviour: str                  # plain-language description
    evidence: dict[str, float] = Field(default_factory=dict)   # metric name -> quoted value (verified by code)
    technique_hints: list[TechniqueHint] = Field(default_factory=list)
    confidence: float = 0.0
    suspicious: bool = True


class Incident(BaseModel):
    id: str
    entity: str
    start: datetime
    end: datetime
    findings: list[Finding]


class GroundTruthSpan(BaseModel):
    """Derived from the dataset labels by the grader only. Agents never see this."""

    label: str
    technique_id: str | None     # via config labels.to_technique
    entity_src: str
    start: datetime
    end: datetime
    flows: int


class EntityStats(BaseModel):
    """Plain statistics for one source address in one window. No judgement, no thresholds."""

    entity: str
    is_private: bool
    flows: int
    dst_ips: int
    dst_ports: int
    top_ports: list[tuple[int, int]]       # (port, flows), at most 3
    ext_dst_ips: int                       # distinct non-private destinations
    mean_duration_ms: float
    fwd_bytes: int
    bwd_bytes: int
    syn: int
    rst: int
    active_minutes: int
    per_minute_mean: float
    per_minute_cv: float                   # 0 = perfectly regular; undefined (0.0) with <2 active minutes
    top_port_share: float = 0.0            # share of this source's flows that hit its most common port
    top_dst_share: float = 0.0             # share of this source's flows that hit its most common destination

    def row(self) -> str:
        ports = ",".join(f"{p}:{n}" for p, n in self.top_ports)
        return (f"src={self.entity} priv={int(self.is_private)} flows={self.flows} dst_ips={self.dst_ips} "
                f"dst_ports={self.dst_ports} top_ports={ports} top_port_share={self.top_port_share:.2f} "
                f"top_dst_share={self.top_dst_share:.2f} ext_dst={self.ext_dst_ips} "
                f"dur_ms={self.mean_duration_ms:.1f} fwd_B={self.fwd_bytes} bwd_B={self.bwd_bytes} "
                f"syn={self.syn} rst={self.rst} act_min={self.active_minutes} "
                f"per_min={self.per_minute_mean:.1f} per_min_cv={self.per_minute_cv:.2f}")


class WindowDigest(BaseModel):
    window_start: datetime
    window_end: datetime
    total_flows: int
    sources: int
    entities: list[EntityStats]
    omitted: int = 0                       # sources not listed individually
    omitted_flows: int = 0
    omitted_max_flows: int = 0
    omitted_max_dst_ports: int = 0
    peers: dict[str, float] = {}           # medians over in-network sources: what 'typical' looks like

    def to_text(self) -> str:
        head = (f"WINDOW {self.window_start:%Y-%m-%d %H:%M}-{self.window_end:%H:%M} "
                f"flows={self.total_flows} sources={self.sources}")
        rows = [e.row() for e in self.entities]
        peers = ""
        if self.peers:
            peers = "PEERS(in-network median) " + " ".join(f"{k}={v:g}" for k, v in self.peers.items())
        tail = (f"UNLISTED sources={self.omitted} flows={self.omitted_flows} "
                f"max_flows_per_source={self.omitted_max_flows} max_dst_ports_per_source={self.omitted_max_dst_ports}")
        return "\n".join(x for x in [head, peers, *rows, tail] if x)


class ProposedAction(BaseModel):
    type: str                               # must be in the action catalog
    target: str                             # ip, domain, host or free text depending on type
    justification: str = ""
    mitigation_id: str | None = None     # official ATT&CK mitigation this action implements


class ResponsePlan(BaseModel):
    incident_id: str
    severity: str = "medium"                # low | medium | high | critical
    risk_score: int = 50                    # 0-100
    rationale: str = ""
    actions: list[ProposedAction] = Field(default_factory=list)


class ActionRecord(BaseModel):
    action: ProposedAction
    status: str                             # executed | denied | rejected
    reason: str = ""
    risk: str = ""


class ActionLog(BaseModel):
    incident_id: str
    records: list[ActionRecord] = Field(default_factory=list)

    def by_status(self, status: str) -> list[ActionRecord]:
        return [r for r in self.records if r.status == status]


class TechniqueAssessment(BaseModel):
    id: str
    confidence: float = 0.5
    citation: str = ""


class ThreatAssessment(BaseModel):
    incident_id: str
    techniques: list[TechniqueAssessment] = Field(default_factory=list)
    rationale: str = ""
    dropped_ids: list[str] = Field(default_factory=list)      # cited but not real / not retrieved


In [ ]:
%%writefile /content/HuntCrew/src/huntcrew/syncutil.py
"""Run blocking agent code safely from notebooks.

Jupyter/Colab already run an asyncio event loop in the main thread, and CrewAI refuses synchronous
`kickoff()` inside a running loop. In that case we execute the call in a short-lived worker thread,
which has no running loop.
"""
from __future__ import annotations

import asyncio
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from typing import TypeVar

T = TypeVar("T")


def run_blocking(fn: Callable[[], T]) -> T:
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return fn()                                   # plain script: no loop, call directly
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(fn).result()


### Scripts
`fetch_data.py` downloads the two external datasets with checksums; `run_eval.py` re-runs the full evaluation (optional, about an hour of API calls).

In [ ]:
%%writefile /content/HuntCrew/scripts/fetch_data.py
"""Download the external datasets into data/external/<name>/ (one new directory per source).

Data is never committed to git; this script + the SHA256 files are the reproducible record.
Standard library only, so it can run unmodified on Colab.

    python -I scripts/fetch_data.py            # fetch everything missing
    python -I scripts/fetch_data.py cic-ids2017
"""
from __future__ import annotations

import hashlib
import sys
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DEST = ROOT / "data" / "external"

SOURCES = {
    "cic-ids2017": {
        "url": "https://huggingface.co/datasets/rdpahalavan/CIC-IDS2017/resolve/main/Network-Flows/CICIDS_Flow.parquet",
        "filename": "CICIDS_Flow.parquet",
        "expected_bytes": 369_814_810,      # as reported by the server on 2026-10-15
        "note": "CIC-IDS2017 flows via HF mirror rdpahalavan/CIC-IDS2017 (apache-2.0 per metadata). "
                "Cite: Sharafaldin, Habibi Lashkari, Ghorbani, ICISSP 2018.",
    },
    "attack-stix": {
        "url": "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json",
        "filename": "enterprise-attack.json",
        "expected_bytes": None,             # unversioned file moves with each ATT&CK release
        "note": "MITRE ATT&CK Enterprise STIX 2.1 (latest release). Subject to ATT&CK Terms of Use.",
    },
}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def fetch(name: str) -> Path:
    src = SOURCES[name]
    out_dir = DEST / name
    target = out_dir / src["filename"]
    if target.exists():
        print(f"[{name}] already present: {target} ({target.stat().st_size:,} bytes)")
        return target
    out_dir.mkdir(parents=True, exist_ok=True)
    part = target.with_suffix(target.suffix + ".part")
    print(f"[{name}] downloading {src['url']}")
    req = urllib.request.Request(src["url"], headers={"User-Agent": "huntcrew-fetch/0.1"})
    with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as f:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    size = part.stat().st_size
    exp = src["expected_bytes"]
    if exp is not None and size != exp:
        part.unlink()
        raise RuntimeError(f"[{name}] size {size:,} != expected {exp:,}; download discarded")
    part.rename(target)
    digest = sha256(target)
    (out_dir / "SHA256SUM").write_text(f"{digest}  {src['filename']}\n{src['note']}\n", encoding="utf-8")
    print(f"[{name}] ok  {size:,} bytes  sha256={digest}")
    return target


if __name__ == "__main__":
    wanted = sys.argv[1:] or list(SOURCES)
    for n in wanted:
        fetch(n)


In [ ]:
%%writefile /content/HuntCrew/scripts/run_eval.py
"""One-command evaluation of B0 / B1 / agent-led / hybrid on a pre-registered split.

    python scripts/run_eval.py dev --systems b0 b1 agent hybrid
    python scripts/run_eval.py test --systems b0 b1 hybrid          # only at evaluation time (ADR-019)

Everything is graded by code against dataset labels. LLM responses are cached, so re-runs are cheap.
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from huntcrew import llm as llmmod  # noqa: E402
from huntcrew.attack_kb import AttackKB, load_chunks  # noqa: E402
from huntcrew.baselines import b0_findings, b1_window, known_destinations  # noqa: E402
from huntcrew.config import load_config  # noqa: E402
from huntcrew.crew import DownstreamCrew, incident_profile  # noqa: E402
from huntcrew.data_cic import load_cic  # noqa: E402
from huntcrew.detector import merge_into_incidents, run_detector  # noqa: E402
from huntcrew.digest import build_digest  # noqa: E402
from huntcrew.evaluate import attack_truth, grade_outcomes, score_findings  # noqa: E402
from huntcrew.pipeline import process_incident  # noqa: E402
from huntcrew.profile import BaselineProfile  # noqa: E402
from huntcrew.replay import Replay  # noqa: E402
from huntcrew.response import Policy, SimulatedEnvironment, auto_deny  # noqa: E402


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("split", choices=["dev", "test"])
    ap.add_argument("--systems", nargs="+", default=["b0", "b1", "hybrid"],
                    choices=["b0", "b1", "agent", "hybrid"])
    ap.add_argument("--names", nargs="*", help="slice names (default: all in the split)")
    ap.add_argument("--max-windows", type=int, help="debug: only the first N windows of each slice")
    ap.add_argument("--no-downstream", action="store_true", help="skip assess/plan/execute/report")
    ap.add_argument("--model")
    ap.add_argument("--out", default=None)
    a = ap.parse_args()

    cfg = load_config()
    llmmod.load_dotenv()
    mw = llmmod.install(cfg["llm"]["min_interval_s"], cfg["llm"]["tokens_per_minute"],
                        cfg["llm"]["max_retries"], cfg["llm"]["cache_dir"])
    llm = llmmod.make_llm(cfg, a.model)
    minutes = cfg["replay"]["window_minutes"]

    data = load_cic(cfg["data"]["cic_parquet"])
    base = cfg["splits"]["baseline_benign"]
    base_flows = data.select(pd.Timestamp(base["start"]), pd.Timestamp(base["end"])).flows
    profile, known = BaselineProfile(base_flows, minutes), known_destinations(base_flows)
    kb = AttackKB(load_chunks(cfg["data"]["attack_stix"]), cfg["rag"]["embedding_model"], cfg["rag"]["persist_dir"])
    policy = Policy.from_file(cfg["data"]["policy"])
    crew = DownstreamCrew(kb, policy, llm)
    kb_ids = set(kb.chunks)

    report: dict = {"split": a.split, "model": a.model or cfg["llm"]["model"], "slices": {}}
    for s in cfg["splits"][a.split]:
        if a.names and s["name"] not in a.names:
            continue
        sl = data.select(pd.Timestamp(s["start"]), pd.Timestamp(s["end"]))
        rp = Replay(sl.flows, s["start"], s["end"], minutes, cfg["replay"]["history_windows"])
        windows = rp.windows()[: a.max_windows] if a.max_windows else rp.windows()
        truth = attack_truth(sl.flows, sl.labels, windows, cfg["labels"]["to_technique"],
                             cfg["grading"]["min_attack_flows"])
        only = [w.index for w in windows]
        out: dict = {"windows": len(windows)}

        for system in a.systems:
            t0, before = time.time(), mw.stats.snapshot()
            if system == "b0":
                findings = [f for w, v in rp if w.index in only
                            for f in b0_findings(v.current(), w, cfg["baseline_rules"], known)]
            elif system == "b1":
                findings = []
                for w, v in rp:
                    if w.index not in only:
                        continue
                    dg = build_digest(v.current(), w, cfg["digest"]["max_private"], cfg["digest"]["public_top_k"])
                    if dg.total_flows:
                        findings += b1_window(llm, dg.to_text(), w, minutes)
            else:
                path = Path(".cache/runs") / f"{'agent' if system == 'agent' else 'hybrid'}_{a.split}_{s['name']}.jsonl"
                res = run_detector(sl.flows, s["start"], s["end"], cfg, kb, profile, llm, path, mw,
                                   only_windows=only, candidates_from="screen" if system == "agent" else "rules",
                                   known_dsts=known)
                findings = [f for r in res for f in r.findings]
            entry = {"detection": {k: v for k, v in score_findings(findings, truth, windows).items()
                                   if k != "per_technique"}}
            incidents = merge_into_incidents(findings, minutes)
            entry["incidents"] = len(incidents)
            if system in {"agent", "hybrid"} and not a.no_downstream:
                env = SimulatedEnvironment()
                outcomes = [process_incident(i, crew, policy, env, auto_deny, write_report=True,
                                             profile=incident_profile(sl.flows, i)) for i in incidents]
                entry["response"] = grade_outcomes(outcomes, truth, windows, kb_ids)
                entry["environment"] = {k: len(v) for k, v in env.snapshot().items()}
            entry["llm"] = mw.stats.delta(before)
            entry["seconds"] = round(time.time() - t0, 1)
            out[system] = entry
            print(f"[{s['name']}] {system}: {json.dumps(entry['detection'])}", flush=True)
        report["slices"][s["name"]] = out

    dest = Path(a.out or f"results/{a.split}_{int(time.time())}.json")
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"wrote {dest}")


if __name__ == "__main__":
    main()


## 3. Download the data and set up
Fetches CIC-IDS2017 flows and the MITRE ATT&CK bundle (about 425 MB, checksum-verified; never stored in the repository), then builds the 706-chunk knowledge base.

In [ ]:
!python -I /content/HuntCrew/scripts/fetch_data.py

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
%cd /content/HuntCrew
sys.path.insert(0, "/content/HuntCrew/src")
from huntcrew import demo
d = demo.setup()          # checks the key, loads data, builds the ATT&CK knowledge base, wires the agents
print("model:", d.cfg["llm"]["model"], "| ATT&CK chunks:", len(d.kb.chunks))

## 4. Replay real traffic through the detector
Windows are revealed one at a time; agents can look back but never forward, and never see labels. Hybrid mode: rules propose candidates, the agent verifies them.

In [ ]:
import pandas as pd
truth = demo.windows_labelled(d, "test", "ssh_patator")        # grader's view, only to pick windows to show
pick = sorted(truth.window_index.unique())[:2]
results, incidents = demo.detect(d, "test", "ssh_patator", windows=pick, mode="rules")
for r in results:
    print(f"window {r.window.label()}: {len(r.findings)} verified finding(s), {len(r.rejected)} rejected")
    for f in r.findings:
        print("  ", f.entity, [h.id for h in f.technique_hints], "|", f.behaviour)

## 5. Name the technique, plan the response, ask before risky actions

In [ ]:
from huntcrew.response import console_approver
outcomes, env = demo.respond(d, incidents, approver=console_approver)   # asks you to approve medium/high risk
for o in outcomes:
    print(o.incident.id, o.incident.entity, "->", [t.id for t in o.assessment.techniques], o.plan.severity)
    for rec in o.log.records:
        print("   ", rec.status.upper(), rec.action.type, rec.action.target, "|", rec.reason)
print("simulated environment:", env.snapshot())

In [ ]:
from IPython.display import Markdown, display
for o in outcomes[:1]:
    display(Markdown(o.report))

## 6. Results (graded by code against dataset labels)
From the one frozen evaluation run on the untouched test slices (SSH-Patator, Bot, PortScan). To re-run it yourself, uncomment the last line (about an hour of API calls).

In [ ]:
import json, pathlib
res = json.loads(pathlib.Path("results/final.json").read_text())
rows = []
for sl, systems in res["slices"].items():
    for sysname, e in systems.items():
        if isinstance(e, dict) and "detection" in e:
            rows.append({"slice": sl, "system": sysname, **{k: e["detection"][k] for k in ("recall", "precision", "f1", "fp")}})
display(pd.DataFrame(rows))
print("pooled over test slices:")
display(pd.DataFrame(res["micro_average"]).T)
print("hybrid, full pipeline:", res["hybrid_response"])
# !python scripts/run_eval.py test --systems b0 b1 hybrid        # optional: re-run the full evaluation